## Импорты

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from copy import deepcopy

import talib

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.ensemble import VotingClassifier
from sklearn.utils.validation import check_is_fitted
from sklearn.model_selection import KFold

from catboost import CatBoostClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from typing import List, Optional, Tuple

## Исходные данные

In [2]:
# Данные по индексу
# https://www.moex.com/ru/index/MOEXBC/archive?from=2009-04-24&till=2024-11-25&sort=TRADEDATE&order=desc

# Курс доллара
# https://investfunds.ru/indexes/39/

!wget 'https://drive.google.com/uc?export=download&id=1QBemvMmFNhiR25JPr_tO-4bYh-1NV0zD' -O 'moexbc.csv'
!wget 'https://drive.google.com/uc?export=download&id=1UYbOHL95Xe0MOEY7VOaTn17sto8MWl3B' -O 'usd_rub-(банк-россии).xlsx'

--2025-08-30 12:18:41--  https://drive.google.com/uc?export=download&id=1QBemvMmFNhiR25JPr_tO-4bYh-1NV0zD
Resolving drive.google.com (drive.google.com)... 173.194.205.138, 173.194.205.100, 173.194.205.139, ...
Connecting to drive.google.com (drive.google.com)|173.194.205.138|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1QBemvMmFNhiR25JPr_tO-4bYh-1NV0zD&export=download [following]
--2025-08-30 12:18:41--  https://drive.usercontent.google.com/download?id=1QBemvMmFNhiR25JPr_tO-4bYh-1NV0zD&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.140.33
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.140.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 648088 (633K) [application/octet-stream]
Saving to: ‘moexbc.csv’

moexbc.csv          100%[===================>] 632,90K  2,00MB/s    in 0,3s    

2025-08-

In [3]:
# Индекс
db = pd.read_csv('moexbc.csv', encoding = 'windows-1251', sep = ';')
db = db.reset_index()
clns = list(db.loc[0, :])
db = db.iloc[1:, :].reset_index(drop=True)
db.columns = clns
db = db[['TRADEDATE','CLOSE','OPEN','HIGH','LOW','VALUE']]
db.columns = ['date', 'close','open', 'high', 'low', 'volume']
for c in ['close','open', 'high', 'low', 'volume']:
    db[c] = db[c].apply(lambda x: float(str(x).replace(',', '.')))
db['date'] = db['date'].apply(lambda x: str(x)[:10])

# Курс валюты
s0 = 'usd_rub-(банк-россии).xlsx'
d0 = pd.read_excel(s0)
d0.columns = ['date', 'USD_RUB']
d0['date'] = d0['date'].apply(lambda x: str(x)[:10])
d0['date'] = d0['date'].apply(lambda x: x[8:10]+'.'+x[5:7]+'.'+x[:4])
d0 = d0[['date', 'USD_RUB']]

# Собираем вместе
db = db.merge(d0, on='date', how='left')
db['close_RUR'] = db['close']
db['close'] = db['close']/db['USD_RUB']


# Обработка даты
c= 'date'
db[c] = db[c].apply(lambda x: x.split('.')[2]+'-'+x.split('.')[1]+'-'+x.split('.')[0])
db[c] = db[c].apply(lambda x: str(x))

c= 'date'
db[c] = db[c].apply(lambda x: str(x))
db = db.interpolate(method='linear').fillna(method='ffill').fillna(method='bfill')
db[c] = db[c].apply(pd.to_datetime, errors='coerce')
db = db.sort_values(c, ascending=True).reset_index(drop=True)

# Перевод всех значений в числовой формат
for c in [x for x in db.columns if x != 'date']:
    db[c] = db[c].apply(pd.to_numeric, errors='coerce')

db

,date,close,open,high,low,volume,USD_RUB,close_RUR
0,2009-04-24,185.766972,6285.76,6362.62,6239.72,8.292833e+08,33.7848,6276.10
1,2009-04-27,181.275454,6276.10,6276.10,6033.84,1.101359e+09,33.4187,6057.99
2,2009-04-28,176.690007,6057.99,6116.65,5885.04,1.978925e+09,33.3904,5899.75
3,2009-04-29,182.067040,5899.75,6111.45,5899.75,2.348292e+09,33.5533,6108.95
4,2009-04-30,188.417130,6108.95,6353.44,6108.95,1.739327e+09,33.2491,6264.70
...,...,...,...,...,...,...,...,...
3894,2024-11-06,174.055185,17171.89,17394.37,16992.03,8.876278e+10,98.0562,17067.19
3895,2024-11-07,175.887363,17067.19,17301.28,16902.70,4.434799e+10,98.2236,17276.29
3896,2024-11-08,178.794281,17488.76,17660.77,17420.44,6.127687e+10,98.0726,17534.82
3897,2024-11-11,182.055329,17839.05,17926.57,17712.09,7.750142e+10,97.8335,17811.11


## Фильтруем данные
классический метод IQR-фильтрации (InterQuartile Range)

В scikit-learn нет готового трансформера “удалить выбросы по IQR”, поэтому напишем кастомный класс-трансформер и упакуем его в sklearn-совместимый трансформер (через BaseEstimator, TransformerMixin), чтобы встроить в Pipeline.

In [4]:
class IQRCarryForwardOutlierRemover(BaseEstimator, TransformerMixin):
    """
    Заменяет выбросы (по правилу IQR) на предыдущее корректное значение.
    - Выбросы: x < Q1 - factor*IQR или x > Q3 + factor*IQR
    - По умолчанию обрабатывает все числовые столбцы, кроме date-колонки.
    - Сохраняет DataFrame и порядок столбцов.
    """
    def __init__(self, factor=1.5, date_col='date', columns=None, sort_by_date=True):
        self.factor = float(factor)
        self.date_col = date_col
        self.columns = columns  # None => авто-выбор числовых
        self.sort_by_date = sort_by_date

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидаю pandas.DataFrame на входе.")
        df = X.copy()

        # авто-выбор колонок: все числовые, кроме date
        if self.columns is None:
            numeric_cols = df.select_dtypes(include=[np.number, "float", "int"]).columns.tolist()
            self.columns_ = [c for c in numeric_cols if c != self.date_col]
        else:
            self.columns_ = list(self.columns)

        # посчитаем пороги по каждому столбцу
        self.bounds_ = {}
        for col in self.columns_:
            s = pd.to_numeric(df[col], errors="coerce")
            q1 = np.nanpercentile(s, 25)
            q3 = np.nanpercentile(s, 75)
            iqr = q3 - q1
            lower = q1 - self.factor * iqr
            upper = q3 + self.factor * iqr
            self.bounds_[col] = (lower, upper)

        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидаю pandas.DataFrame на входе.")
        if not hasattr(self, "bounds_"):
            raise RuntimeError("Сначала вызовите fit().")

        df = X.copy()

        # сортировка по дате (если нужно) — важна для "последнего корректного"
        if self.sort_by_date and self.date_col in df.columns:
            df = df.sort_values(self.date_col).reset_index(drop=True)

        for col in self.columns_:
            s = pd.to_numeric(df[col], errors="coerce")

            lower, upper = self.bounds_[col]
            is_outlier = (s < lower) | (s > upper)

            # выбросы -> NaN, затем тянем последнее корректное значение вперёд
            s_masked = s.mask(is_outlier)
            s_filled = s_masked.ffill()

            # если выбросы в самом начале (нет "прошлого" значения) — оставляем исходные
            leading = s_filled.isna()
            if leading.any():
                s_filled[leading] = s[leading]

            df[col] = s_filled

        return df

In [5]:

pipe_filter = Pipeline([
    ("iqr_cf", IQRCarryForwardOutlierRemover(
        factor=1.5,
        date_col='date',
        columns=['close','open','high','low','volume','USD_RUB','close_RUR'],  # можно не указывать: выберет числовые сам
        sort_by_date=True
    ))
])

df_clean = pipe_filter.fit_transform(db)  # db — ваш исходный DataFrame

## Создание признаков

In [6]:
# RSI

def calculate_rsi(new_data: pd.DataFrame, column='close', window=14)->pd.Series:
    delta = new_data[column].diff()
    gain = (delta.where(delta > 0, 0)).fillna(0)
    loss = (-delta.where(delta < 0, 0)).fillna(0)
    
    avg_gain = gain.rolling(window=window, min_periods=1).mean()
    avg_loss = loss.rolling(window=window, min_periods=1).mean()
    
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    rsi.fillna(0, inplace=True)
    return rsi

In [7]:
# Moving RSI
def calculate_moving_rsi(rsi: pd.Series, window=14):
    return rsi.rolling(window=window, min_periods=1).mean()

In [8]:
class TechIndicatorsTransformer(BaseEstimator, TransformerMixin):
    """Добавляет новые фичи:
    - RSI и его скользящее среднне
    - MACD, MACD_Signal, MACD_Hist
    - SMA короткое идлинное
    Args:
        BaseEstimator (_type_): _description_
        TransformerMixin (_type_): _description_
    """
    def __init__(
        self,
        price_col="close",
        rsi_window=14,
        rsi_ma_window=14,
        sma_short=20,
        sma_long=50,
        date_col="date",
        sort_by_date=True,
    ):
        self.price_col = price_col
        self.rsi_window = rsi_window
        self.rsi_ma_window = rsi_ma_window
        self.sma_short = sma_short
        self.sma_long = sma_long
        self.date_col = date_col
        self.sort_by_date = sort_by_date

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("TechnicalIndicatorsTransformer ожидает pandas.DataFrame")

        df = X.copy()

        # сортировка по дате (для временного ряда)
        if self.sort_by_date and self.date_col in df.columns:
            df = df.sort_values(self.date_col).reset_index(drop=True)

        price = df[self.price_col].astype(float).values

        # RSI
        rsi = calculate_rsi(df, column=self.price_col, window=self.rsi_window)
        df[f"RSI_{self.rsi_window}"] = rsi
        df[f"RSI_{self.rsi_window}_MA{self.rsi_ma_window}"] = calculate_moving_rsi(
            rsi, window=self.rsi_ma_window
        )

        # MACD
        macd, macd_signal, macd_hist = talib.MACD(
            price, fastperiod=12, slowperiod=26, signalperiod=9
        )
        df["MACD"] = macd
        df["MACD_Signal"] = macd_signal
        df["MACD_Hist"] = macd_hist
        
        # Рассчитываем TripleEMA и MACD
        df['tema'] = talib.TEMA(df['close'], timeperiod=24)
        df['macd'], df['macd_signal'], df['macd_hist'] = talib.MACD(df['close'], fastperiod=12, slowperiod=26, signalperiod=9)

        # SMA short/long
        df[f"SMA_{self.sma_short}"] = talib.SMA(price, timeperiod=self.sma_short)
        df[f"SMA_{self.sma_long}"] = talib.SMA(price, timeperiod=self.sma_long)

        df.fillna(0, inplace=True)
        
        return df

In [9]:
pipe_transform = Pipeline([
    ("ti", TechIndicatorsTransformer(
        price_col="close",
        rsi_window=14,
        rsi_ma_window=14,
        sma_short=20,
        sma_long=50,
    ))
])

df_new = pipe_transform.fit_transform(df_clean)  # db = DataFrame с колонками ['date','close','open','high','low',...]
df_new.head()

,date,close,open,high,low,volume,USD_RUB,close_RUR,RSI_14,RSI_14_MA14,MACD,MACD_Signal,MACD_Hist,tema,macd,macd_signal,macd_hist,SMA_20,SMA_50
0,2009-04-24,185.766972,6285.76,6362.62,6239.72,8.292833e+08,33.7848,6276.10,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2009-04-27,181.275454,6276.10,6276.10,6033.84,1.101359e+09,33.4187,6057.99,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2009-04-28,176.690007,6057.99,6116.65,5885.04,1.978925e+09,33.3904,5899.75,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2009-04-29,182.067040,5899.75,6111.45,5899.75,2.348292e+09,33.5533,6108.95,37.201007,9.300252,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2009-04-30,188.417130,6108.95,6353.44,6108.95,1.739327e+09,33.2491,6264.70,56.369320,18.714065,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Модель на базе тех анализа

In [10]:
def _ema(s: pd.Series, period: int) -> pd.Series:
    return pd.Series(s, index=s.index, dtype=float).ewm(span=period, adjust=False).mean()

def _tema(close: pd.Series, period: int) -> pd.Series:
    # TEMA = 3*EMA1 - 3*EMA2 + EMA3
    ema1 = _ema(close, period)
    ema2 = _ema(ema1, period)
    ema3 = _ema(ema2, period)
    return 3*ema1 - 3*ema2 + ema3

def _macd_from_close(close: pd.Series, fast=12, slow=26, signal=9):
    ema_fast = _ema(close, fast)
    ema_slow = _ema(close, slow)
    macd = ema_fast - ema_slow
    macd_signal = _ema(macd, signal)
    return macd, macd_signal

class RuleSignalClassifier(BaseEstimator, ClassifierMixin):
    """
    Классификатор по правилу:
      buy  (1):  macd > macd_signal  AND  close > tema
      sell (-1): macd < macd_signal  AND  close < tema
      hold (0):  иначе

    Параметры:
      close_col: имя колонки цены закрытия
      macd_col, macd_signal_col, tema_col: имена колонок с индикаторами, если уже есть
      compute_if_missing: считать индикаторы из close, если колонок нет
      macd_fast, macd_slow, macd_signal: параметры MACD, если считаем
      tema_period: период TEMA, если считаем
      neutral_class: метка «нет сигнала» (по умолчанию 0)
    """

    
    def __init__(self,
                 close_col="close",
                 macd_col="macd",
                 macd_signal_col="macd_signal",
                 tema_col="tema",
                 compute_if_missing=True,
                 macd_fast=12, macd_slow=26, macd_signal=9,
                 tema_period=30,
                 neutral_class=0):
        self.close_col = close_col
        self.macd_col = macd_col
        self.macd_signal_col = macd_signal_col
        self.tema_col = tema_col
        self.compute_if_missing = compute_if_missing
        self.macd_fast = macd_fast
        self.macd_slow = macd_slow
        self.macd_signal = macd_signal
        self.tema_period = tema_period
        self.neutral_class = neutral_class

    # обучаться тут нечему — но sklearn ожидает fit
    def fit(self, X: pd.DataFrame, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame")
        # объявим порядок классов для predict_proba
        self.classes_ = np.array([-1, self.neutral_class, 1], dtype=int)
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        df = self._ensure_indicators(X)
        macd = pd.to_numeric(df[self.macd_col], errors="coerce")
        macd_signal = pd.to_numeric(df[self.macd_signal_col], errors="coerce")
        close = pd.to_numeric(df[self.close_col], errors="coerce")
        tema = pd.to_numeric(df[self.tema_col], errors="coerce")

        # условия
        buy = (macd > macd_signal) & (close > tema)
        sell = (macd < macd_signal) & (close < tema)

        out = np.full(len(df), self.neutral_class, dtype=int)
        out[sell.fillna(False).values] = -1
        out[buy.fillna(False).values] = 1
        return out

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """
        Жёсткие «вероятности»: 1.0 для предсказанного класса, 0.0 для остальных.
        Нужно лишь для совместимости со scorer’ами, если потребуется.
        """
        y = self.predict(X)
        proba = np.zeros((len(y), 3), dtype=float)  # порядок: [-1, neutral, +1]
        for i, label in enumerate(y):
            if label == -1:
                proba[i, 0] = 1.0
            elif label == 1:
                proba[i, 2] = 1.0
            else:
                proba[i, 1] = 1.0
        return proba

    # --- вспомогательное ---
    def _ensure_indicators(self, X: pd.DataFrame) -> pd.DataFrame:
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame")
        if self.close_col not in X.columns:
            raise KeyError(f"Не найдена колонка '{self.close_col}'")

        df = X.copy()

        # MACD
        need_macd = (self.macd_col not in df.columns) or (self.macd_signal_col not in df.columns)
        if need_macd:
            if not self.compute_if_missing:
                missing = [c for c in [self.macd_col, self.macd_signal_col] if c not in df.columns]
                raise KeyError(f"Отсутствуют {missing}, а вычислять запрещено (compute_if_missing=False)")
            macd, macd_sig = _macd_from_close(
                df[self.close_col].astype(float),
                fast=self.macd_fast, slow=self.macd_slow, signal=self.macd_signal
            )
            df[self.macd_col] = macd
            df[self.macd_signal_col] = macd_sig

        # TEMA
        if self.tema_col not in df.columns:
            if not self.compute_if_missing:
                raise KeyError(f"Отсутствует '{self.tema_col}', а вычислять запрещено (compute_if_missing=False)")
            df[self.tema_col] = _tema(df[self.close_col].astype(float), self.tema_period)

        return df

In [11]:
rule_clf = RuleSignalClassifier(
    close_col="close",
    macd_col="macd",
    macd_signal_col="macd_signal",
    tema_col="tema",
    compute_if_missing=True,  # посчитает индикаторы из close при отсутствии
    macd_fast=12, macd_slow=26, macd_signal=9,
    tema_period=30,
    neutral_class=0
)

In [12]:
# Итоговый пайплайн
pipe = Pipeline([
    ('filter', pipe_filter),
    ('transformer', pipe_transform),
    ("rule_model", rule_clf),
])

pipe

Pipeline(steps=[('filter',
                 Pipeline(steps=[('iqr_cf',
                                  IQRCarryForwardOutlierRemover(columns=['close',
                                                                         'open',
                                                                         'high',
                                                                         'low',
                                                                         'volume',
                                                                         'USD_RUB',
                                                                         'close_RUR']))])),
                ('transformer',
                 Pipeline(steps=[('ti', TechIndicatorsTransformer())])),
                ('rule_model', RuleSignalClassifier())])

In [13]:
pipe.fit(db)   # db — ваш DataFrame
signals = pipe.predict(db)

In [14]:
pipe

Pipeline(steps=[('filter',
                 Pipeline(steps=[('iqr_cf',
                                  IQRCarryForwardOutlierRemover(columns=['close',
                                                                         'open',
                                                                         'high',
                                                                         'low',
                                                                         'volume',
                                                                         'USD_RUB',
                                                                         'close_RUR']))])),
                ('transformer',
                 Pipeline(steps=[('ti', TechIndicatorsTransformer())])),
                ('rule_model', RuleSignalClassifier())])

## Добавим еще один классификатор в модель

In [15]:
# ---------- DataFrame-обёртка над MinMaxScaler (сохраняем DataFrame-формат) ----------
class DataFrameMinMaxScaler(BaseEstimator, TransformerMixin):
    """
    Масштабирует указанные столбцы DataFrame с сохранением формата DataFrame.
    По умолчанию: скейлит все числовые колонки, КРОМЕ price_col и date_col.
    """
    def __init__(self, columns=None, price_col="close", date_col="date", feature_range=(0,1), clip=False):
        self.columns = columns
        self.price_col = price_col
        self.date_col = date_col
        self.feature_range = feature_range
        self.clip = clip

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame")
        df = X

        if self.columns is None:
            numeric = df.select_dtypes(include=[np.number, "float", "int"]).columns.tolist()
            # исключим цену и дату по умолчанию
            self.columns_ = [c for c in numeric if c not in {self.price_col, self.date_col}]
        else:
            self.columns_ = list(self.columns)

        self.scaler_ = MinMaxScaler(feature_range=self.feature_range, clip=self.clip)
        self.scaler_.fit(df[self.columns_].values)
        return self

    def transform(self, X):
        df = X.copy()
        arr = self.scaler_.transform(df[self.columns_].values)
        df.loc[:, self.columns_] = arr
        return df

In [16]:
class CatBoostSignalClassifier(BaseEstimator, ClassifierMixin):
    """
    y[t] =  1, если close[t+1] > close[t] + const_margin
           -1, если close[t+1] < close[t] - const_margin
            0, иначе
    Обучается CatBoost на числовых фичах X (можно уже отмасштабированных шагом до него).
    """
    
    def __init__(self,
                 price_col="close",
                 const_margin=0.0,
                 feature_cols=None,      # если None — все числовые, кроме date
                 date_col="date",
                 sort_by_date=True,
                 cat_params=None,
                 neutral_class=0):
        self.price_col = price_col
        self.const_margin = float(const_margin)
        self.feature_cols = feature_cols
        self.date_col = date_col
        self.sort_by_date = sort_by_date
        self.cat_params = cat_params or {}
        self.neutral_class = neutral_class

    def _build_target(self, close: pd.Series) -> pd.Series:
        nxt = close.shift(-1)
        diff = nxt - close
        y = pd.Series(self.neutral_class, index=close.index, dtype=int)
        y[diff >  self.const_margin] = 1
        y[diff < -self.const_margin] = -1
        return y  # последний индекс останется neutral (нет next)

    def _select_feature_cols(self, df: pd.DataFrame):
        if self.feature_cols is not None:
            return list(self.feature_cols)
        cols = df.select_dtypes(include=[np.number, "float", "int"]).columns.tolist()
        self.feature_cols_ = [c for c in cols if c != self.date_col]  # close остаётся как фича — обычно это ок
        return self.feature_cols_

    def fit(self, X: pd.DataFrame, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame")
        df = X.copy()
        if self.sort_by_date and self.date_col in df.columns:
            df = df.sort_values(self.date_col).reset_index(drop=True)

        if self.price_col not in df.columns:
            raise KeyError(f"Нет '{self.price_col}' для построения цели")

        # целевая метка из НЕскейленного close (мы его не трогаем в скейлере)
        y_full = self._build_target(pd.to_numeric(df[self.price_col], errors="coerce"))

        # фичи
        feat_cols = self._select_feature_cols(df)
        X_full = df[feat_cols].copy()

        mask = X_full.notna().all(axis=1) & y_full.notna()
        X_train = X_full[mask]
        y_train = y_full[mask]

        # маппинг классов {-1,0,1} -> {0,1,2}
        self.classes_ = np.array([-1, self.neutral_class, 1], dtype=int)
        label_to_idx = {-1:0, self.neutral_class:1, 1:2}
        y_idx = y_train.map(label_to_idx).astype(int).values

        params = dict(
            loss_function="MultiClass",
            iterations=500,
            depth=6,
            learning_rate=0.05,
            random_seed=42,
            verbose=False
        )
        params.update(self.cat_params or {})

        self.model_ = CatBoostClassifier(**params)
        self.model_.fit(X_train.values, y_idx)

        self.feature_cols_ = X_train.columns.tolist()
        return self

    def _prepare_features(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.sort_by_date and self.date_col in X.columns:
            X = X.sort_values(self.date_col).reset_index(drop=True)
        feats = X[self.feature_cols_].copy()
        feats = feats.fillna(method="ffill").fillna(method="bfill")
        return feats

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        feats = self._prepare_features(X)
        y_idx_pred = self.model_.predict(feats.values).ravel().astype(int)
        idx_to_label = {0:-1, 1:self.neutral_class, 2:1}
        return np.vectorize(idx_to_label.get)(y_idx_pred)

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        feats = self._prepare_features(X)
        return self.model_.predict_proba(feats.values)

In [17]:
# --- трансформеры ---
tech = TechIndicatorsTransformer(
    price_col="close",
    rsi_window=14, rsi_ma_window=14,
    sma_short=20, sma_long=50
)

scaler = DataFrameMinMaxScaler(
    columns=None,        # авто: все числовые, кроме 'close' и 'date'
    price_col="close",
    date_col="date"
)

# --- классификаторы ---
rule = RuleSignalClassifier(
    close_col="close",
    macd_col="macd",
    macd_signal_col="macd_signal",
    tema_col="tema",
    compute_if_missing=True,
    tema_period=30,
    neutral_class=0
)

cat = CatBoostSignalClassifier(
    price_col="close",
    const_margin=0.001,
    feature_cols=None,
    date_col="date",
    sort_by_date=True,
    cat_params=dict(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        random_seed=42,
        verbose=False,
        loss_function="MultiClass",
    ),
    neutral_class=0
)

# --- пайплайны для каждой ветви ---
pipe_cat = Pipeline([
    ("tech", tech),
    ("scaler", scaler),
    ("cat", cat)
])


pipe_rule = Pipeline([
    ("tech", tech),
    ("rule", rule)
])


# --- финальный VotingClassifier ---
pipe = VotingClassifier(
    estimators=[
        ("cat_branch", pipe_cat),
        ("rule_branch", pipe_rule)
    ],
    voting="soft",         # усредняем вероятности
    weights=[0.7, 0.3]     # можно регулировать
)


pipe

VotingClassifier(estimators=[('cat_branch',
                              Pipeline(steps=[('tech',
                                               TechIndicatorsTransformer()),
                                              ('scaler',
                                               DataFrameMinMaxScaler()),
                                              ('cat',
                                               CatBoostSignalClassifier(cat_params={'depth': 6,
                                                                                    'iterations': 500,
                                                                                    'learning_rate': 0.05,
                                                                                    'loss_function': 'MultiClass',
                                                                                    'random_seed': 42,
                                                                                    'verbose': False},
                                                                        const_margin=0.001))])),
                             ('rule_branch',
                              Pipeline(steps=[('tech',
                                               TechIndicatorsTransformer()),
                                              ('rule',
                                               RuleSignalClassifier())]))],
                 voting='soft', weights=[0.7, 0.3])

In [18]:
# Проблема 1 - VoitingClassifier требует y

pipe.fit(db)

TypeError: VotingClassifier.fit() missing 1 required positional argument: 'y'

<div class="alert alert-warning">

<b>Ошибка</b>

Это из-за VotingClassifier: он — supervised и требует `y` при `fit`. При вызове `pipe.fit(db)` без целевого вектора он падает с ValueError: Expected array-like … got None.

У нас целевая метка по правилу `close[t+1] vs close[t] ± const`. Построим её заранее и обучим пайплайн на `X` без последней строки (для последнего дня нет «следующего» значения).
    
</div>

In [19]:
CONST_MARGIN = 0.001  # ваш порог в единицах цены (или задайте свой)

def build_target(df: pd.DataFrame, price_col="close", const_margin=CONST_MARGIN) -> pd.Series:
    nxt = df[price_col].shift(-1)
    diff = nxt - df[price_col]
    y = np.where(diff >  const_margin,  1,
        np.where(diff < -const_margin, -1, 0))
    # Последняя строка не имеет "nxt" → убираем
    return pd.Series(y[:-1], index=df.index[:-1], dtype=int)

# 1) Формируем y
y = build_target(db, price_col="close", const_margin=CONST_MARGIN)

# 2) Подготовим X под ту же длину
X_train = db.iloc[:-1].copy()

In [20]:
# Проблема 2 - Все шаги внутри voitingClassifier должны быть классификаторами

pipe.fit(X_train, y)

ValueError: The estimator Pipeline should be a classifier.

<div class="alert alert-warning">

<b>Ошибка</b>

Все шаги внутри voitingClassifier должны быть классификаторами
    
</div>

In [21]:
# Create a custom ensemble class that doesn't rely on VotingClassifier
class CustomEnsembleClassifier(BaseEstimator, ClassifierMixin):
    """
    Custom ensemble classifier that combines rule-based and ML approaches
    """
    
    def __init__(self, cat_pipeline, rule_pipeline, weights=None):
        self.cat_pipeline = cat_pipeline
        self.rule_pipeline = rule_pipeline
        self.weights = weights or [0.7, 0.3]
        
    def fit(self, X, y=None):
        # Fit both pipelines
        self.cat_pipeline.fit(X, y)
        self.rule_pipeline.fit(X, y)
        
        # Set classes
        self.classes_ = np.array([-1, 0, 1])
        
        return self
    
    def predict_proba(self, X):
        # Get probabilities from both models
        cat_proba = self.cat_pipeline.predict_proba(X)
        rule_proba = self.rule_pipeline.predict_proba(X)
        
        # Weighted average
        ensemble_proba = (self.weights[0] * cat_proba + 
                         self.weights[1] * rule_proba)
        
        return ensemble_proba
    
    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]

In [22]:
# Create custom ensemble
ensemble = CustomEnsembleClassifier(
    cat_pipeline=pipe_cat,
    rule_pipeline=pipe_rule,
    weights=[0.7, 0.3]
    )

ensemble

CustomEnsembleClassifier(cat_pipeline=Pipeline(steps=[('tech',
                                                       TechIndicatorsTransformer()),
                                                      ('scaler',
                                                       DataFrameMinMaxScaler()),
                                                      ('cat',
                                                       CatBoostSignalClassifier(cat_params={'depth': 6,
                                                                                            'iterations': 500,
                                                                                            'learning_rate': 0.05,
                                                                                            'loss_function': 'MultiClass',
                                                                                            'random_seed': 42,
                                                                                            'verbose': False},
                                                                                const_margin=0.001))]),
                         rule_pipeline=Pipeline(steps=[('tech',
                                                        TechIndicatorsTransformer()),
                                                       ('rule',
                                                        RuleSignalClassifier())]),
                         weights=[0.7, 0.3])

In [23]:
ensemble.fit(X_train, y)

CustomEnsembleClassifier(cat_pipeline=Pipeline(steps=[('tech',
                                                       TechIndicatorsTransformer()),
                                                      ('scaler',
                                                       DataFrameMinMaxScaler()),
                                                      ('cat',
                                                       CatBoostSignalClassifier(cat_params={'depth': 6,
                                                                                            'iterations': 500,
                                                                                            'learning_rate': 0.05,
                                                                                            'loss_function': 'MultiClass',
                                                                                            'random_seed': 42,
                                                                                            'verbose': False},
                                                                                const_margin=0.001))]),
                         rule_pipeline=Pipeline(steps=[('tech',
                                                        TechIndicatorsTransformer()),
                                                       ('rule',
                                                        RuleSignalClassifier())]),
                         weights=[0.7, 0.3])

In [24]:
ensemble.predict(X_train)

array([-1, -1, -1, ...,  1,  1,  1])

## Неросетевая модель - LSTM

### Подготовим данные

In [25]:
class Trarget(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        price_col="close",
        date_col="date",
        const_margin = 0.001,  # ваш порог в единицах цены (или задайте свой)
        sort_by_date=True,
    ):
        self.price_col = price_col
        self.date_col = date_col
        self.const_margin = const_margin
        self.sort_by_date = sort_by_date
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("TechnicalIndicatorsTransformer ожидает pandas.DataFrame")

        df = X.copy()

        # сортировка по дате (для временного ряда)
        if self.sort_by_date and self.date_col in df.columns:
            df = df.sort_values(self.date_col).reset_index(drop=True)

        price = df[self.price_col].astype(float).values
        
        # build_target
        nxt = df['close'].shift(-1)
        diff = nxt - df['close']
        y = np.where(diff >  self.const_margin,  1,
            np.where(diff < -self.const_margin, -1, 0))
        df['target'] = y
        
        return df

In [26]:

pipe = Pipeline([
    ('filter', pipe_filter),
    ('transformer', pipe_transform),
    ('target', Trarget(
        price_col='close',
        date_col='date',
        const_margin=0.02
    ))
])

pipe.fit(db)

df = pipe.transform(db)

df.head()

,date,close,open,high,low,volume,USD_RUB,close_RUR,RSI_14,RSI_14_MA14,MACD,MACD_Signal,MACD_Hist,tema,macd,macd_signal,macd_hist,SMA_20,SMA_50,target
0,2009-04-24,185.766972,6285.76,6362.62,6239.72,8.292833e+08,33.7848,6276.10,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1
1,2009-04-27,181.275454,6276.10,6276.10,6033.84,1.101359e+09,33.4187,6057.99,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1
2,2009-04-28,176.690007,6057.99,6116.65,5885.04,1.978925e+09,33.3904,5899.75,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,2009-04-29,182.067040,5899.75,6111.45,5899.75,2.348292e+09,33.5533,6108.95,37.201007,9.300252,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,2009-04-30,188.417130,6108.95,6353.44,6108.95,1.739327e+09,33.2491,6264.70,56.369320,18.714065,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [27]:
# У нас в начале 50 пустых признаков - выбросим их

df = df.iloc[50:]
df.head()

,date,close,open,high,low,volume,USD_RUB,close_RUR,RSI_14,RSI_14_MA14,MACD,MACD_Signal,MACD_Hist,tema,macd,macd_signal,macd_hist,SMA_20,SMA_50,target
50,2009-07-08,193.379622,6286.77,6286.77,6035.60,7.673842e+09,31.4695,6085.56,28.998972,25.861006,-6.227750,-3.131067,-3.096683,0.0,-6.227750,-3.131067,-3.096683,217.629568,219.447253,-1
51,2009-07-09,190.442359,6087.77,6170.19,5991.19,8.594343e+09,31.7819,6052.62,23.745222,24.564865,-7.227091,-3.950272,-3.276819,0.0,-7.227091,-3.950272,-3.276819,214.818117,219.630591,-1
52,2009-07-10,185.903073,6054.52,6075.39,5873.15,8.336038e+09,31.8878,5928.04,27.589617,25.013915,-8.289799,-4.818178,-3.471622,0.0,-8.289799,-4.818178,-3.471622,211.488455,219.814853,1
53,2009-07-13,187.805327,5928.04,6061.96,5790.45,1.131288e+10,32.0353,6016.40,34.558993,26.327225,-8.876188,-5.629780,-3.246408,0.0,-8.876188,-5.629780,-3.246408,208.580936,219.929618,-1
54,2009-07-14,185.977489,5984.47,6224.79,5906.03,1.285993e+10,33.0597,6148.36,21.856489,26.080453,-9.380267,-6.379877,-3.000390,0.0,-9.380267,-6.379877,-3.000390,205.909055,219.880826,1


### Построим LSTM модель

In [28]:
# ---------------------- вспомогательные вещи ----------------------

def _infer_feature_cols(df: pd.DataFrame, target_col: str) -> List[str]:
    exclude = {target_col, "date"}
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        dt = df[c].dtype
        try:
            is_num = np.issubdtype(dt, np.number)
        except TypeError:
            is_num = False
        if is_num:
            cols.append(c)
    return cols

def _build_sequences(X: np.ndarray, y: Optional[np.ndarray], lookback: int, horizon: int):
    n = len(X)
    last_start = n - lookback - horizon + 1
    if last_start <= 0:
        raise ValueError("Недостаточно строк в X для заданных lookback/horizon.")
    X_seq = np.stack([X[i:i+lookback] for i in range(last_start)])
    y_seq = None if y is None else np.asarray([y[i+lookback+horizon-1] for i in range(last_start)])
    return X_seq, y_seq

class _SeqDS(Dataset):
    def __init__(self, X_seq: np.ndarray, y_seq: Optional[np.ndarray] = None):
        self.X = torch.from_numpy(X_seq.astype(np.float32))
        self.y = None if y_seq is None else torch.from_numpy(y_seq).long()
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        if self.y is None: return self.X[i]
        return self.X[i], self.y[i]

class _LSTM(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=2, dropout=0.2, bidirectional=False, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=hidden_size, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers>1 else 0.0, bidirectional=bidirectional
        )
        d = 2 if bidirectional else 1
        self.head = nn.Sequential(
            nn.LayerNorm(d*hidden_size),
            nn.Linear(d*hidden_size, d*hidden_size), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d*hidden_size, num_classes)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(last)

def _class_weights(y: np.ndarray, n_classes: int) -> torch.Tensor:
    counts = np.bincount(y, minlength=n_classes).astype(np.float32)
    counts[counts==0] = 1.0
    inv = 1.0 / counts
    w = inv * (n_classes / inv.sum())
    return torch.tensor(w, dtype=torch.float32)

In [29]:
# ---------------------- основной класс ----------------------

class TimeSeriesLSTMClassifier(BaseEstimator, ClassifierMixin):
    """
    sklearn-совместимый LSTM классификатор для временных рядов.
    Совместим с VotingClassifier (поддерживает predict и predict_proba).

    Важно: возвращает предсказания для каждой строки X.
    Для первых (lookback+horizon-1) строк применяется заполнение (pad_strategy):
      - 'edge': повторяется первое доступное предсказание;
      - 'constant': константа pad_value (класс) для первых шагов.
    """
    def __init__(
        self,
        feature_cols: Optional[List[str]] = None,
        target_col: str = "target",
        lookback: int = 64,
        horizon: int = 1,
        hidden_size: int = 64,
        num_layers: int = 2,
        dropout: float = 0.2,
        bidirectional: bool = False,
        batch_size: int = 256,
        max_epochs: int = 30,
        lr: float = 1e-3,
        weight_decay: float = 1e-4,
        patience: int = 7,
        device: Optional[str] = None,
        num_workers: int = 0,
        pad_strategy: str = "edge",   # 'edge' | 'constant'
        pad_value: int = 0,
        verbose: int = 1
    ):
        self.feature_cols = list(feature_cols) if feature_cols is not None else None
        self.target_col = target_col
        self.lookback = lookback
        self.horizon = horizon
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout
        self.bidirectional = bidirectional
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.lr = lr
        self.weight_decay = weight_decay
        self.patience = patience
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.num_workers = num_workers
        self.pad_strategy = pad_strategy
        self.pad_value = pad_value
        self.verbose = verbose

    def fit(self, X: pd.DataFrame, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame с колонкой 'date' и числовыми фичами.")

        # Сортировка и единый порядок для X и y
        if "date" in X.columns:
            order = np.argsort(pd.to_datetime(X["date"]).values)
            X = X.iloc[order].reset_index(drop=True)
            if y is not None:
                if isinstance(y, (pd.Series, pd.DataFrame, pd.Index)):
                    y = np.asarray(y.iloc[order]).ravel()
                else:
                    y = np.asarray(y).ravel()
        else:
            # если нет date — просто выровняем индексы и приведём y к массиву
            if y is not None and isinstance(y, (pd.Series, pd.DataFrame, pd.Index)):
                y = np.asarray(y).ravel()

        # если y не передан — берём из X[target_col]
        if y is None:
            if self.target_col not in X.columns:
                raise ValueError("y не передан и столбец target_col отсутствует в X.")
            y = X[self.target_col].to_numpy().ravel()

        # фичи
        feat_cols = self.feature_cols or _infer_feature_cols(X, self.target_col)
        Xf = X[feat_cols].to_numpy(dtype=np.float32)

        # масштабирование
        self.scaler_ = StandardScaler()
        Xs = self.scaler_.fit_transform(Xf)

        # кодирование меток
        self.le_ = LabelEncoder()
        y_enc = self.le_.fit_transform(np.asarray(y).ravel())
        self.classes_ = self.le_.classes_
        n_classes = len(self.classes_)

        # последовательности
        X_seq, y_seq = _build_sequences(Xs, y_enc, self.lookback, self.horizon)
        ds = _SeqDS(X_seq, y_seq)
        dl = DataLoader(ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

        # модель
        n_features = X_seq.shape[-1]
        self.model_ = _LSTM(
            n_features=n_features, hidden_size=self.hidden_size, num_layers=self.num_layers,
            dropout=self.dropout, bidirectional=self.bidirectional, num_classes=n_classes
        ).to(self.device)
        weights = _class_weights(y_seq, n_classes).to(self.device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        optim = torch.optim.AdamW(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, mode="min", factor=0.5, patience=3, verbose=False)

        best_loss = float("inf")
        best_state = None
        patience_left = self.patience

        for epoch in range(1, self.max_epochs+1):
            self.model_.train()
            losses = []
            for xb, yb in dl:
                xb = xb.to(self.device); yb = yb.to(self.device)
                optim.zero_grad()
                logits = self.model_(xb)
                loss = criterion(logits, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model_.parameters(), 1.0)
                optim.step()
                losses.append(loss.item())
            val_loss = float(np.mean(losses)) if losses else np.nan
            scheduler.step(val_loss)
            if self.verbose:
                print(f"Epoch {epoch:03d} | loss {val_loss:.4f}")
            if val_loss < best_loss - 1e-4:
                best_loss = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.model_.state_dict().items()}
                patience_left = self.patience
            else:
                patience_left -= 1
                if patience_left <= 0:
                    if self.verbose: print("Early stopping")
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state)
        self.feature_cols_ = feat_cols
        return self

    def predict(self, X: pd.DataFrame):
        proba = self.predict_proba(X)
        preds_enc = np.argmax(proba, axis=1)
        preds = self.le_.inverse_transform(preds_enc)
        return preds

    def predict_proba(self, X: pd.DataFrame):
        check_is_fitted(self, ["model_", "scaler_", "le_", "feature_cols_"])
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame с колонкой 'date' и числовыми фичами.")
        if "date" in X.columns:
            X = X.sort_values("date").reset_index(drop=True)

        Xf = X[self.feature_cols_].to_numpy(dtype=np.float32)
        Xs = self.scaler_.transform(Xf)

        need = self.lookback + self.horizon - 1
        C = len(self.classes_)

        if len(Xs) < need:
            return np.full((len(Xs), C), 1.0 / C, dtype=np.float32)

        X_seq, _ = _build_sequences(Xs, None, self.lookback, self.horizon)
        ds = _SeqDS(X_seq)
        dl = DataLoader(ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

        self.model_.eval()
        probs_seq = []
        with torch.no_grad():
            for xb in dl:
                xb = xb.to(self.device)
                logits = self.model_(xb)
                p = torch.softmax(logits, dim=1).cpu().numpy()
                probs_seq.append(p)
        probs_seq = np.concatenate(probs_seq, axis=0)  # (len(X)-need+1, C)

        pad_len = need - 1
        if self.pad_strategy == "edge":
            pad_block = np.repeat(probs_seq[[0]], pad_len, axis=0)
        elif self.pad_strategy == "constant":
            pad_block = np.zeros((pad_len, C), dtype=probs_seq.dtype)
            where = np.where(self.classes_ == self.pad_value)[0]
            if len(where) == 0:
                pad_block[:] = 1.0 / C
            else:
                pad_block[:, where[0]] = 1.0
        else:
            raise ValueError("pad_strategy должен быть 'edge' или 'constant'.")

        probs = np.vstack([pad_block, probs_seq])  # (len(X), C)
        return probs

In [30]:
df.columns[1:-1]

Index(['close', 'open', 'high', 'low', 'volume', 'USD_RUB', 'close_RUR',
       'RSI_14', 'RSI_14_MA14', 'MACD', 'MACD_Signal', 'MACD_Hist', 'tema',
       'macd', 'macd_signal', 'macd_hist', 'SMA_20', 'SMA_50'],
      dtype='object')

In [31]:
clf_lstm = TimeSeriesLSTMClassifier(
    feature_cols=df.columns[1:-1],
    target_col='target',
    lookback=64, horizon=1, max_epochs=25, verbose=0
)

In [32]:
clf_lstm.fit(df, df['target'])          # Важно: передаём DataFrame с колонкой date

TimeSeriesLSTMClassifier(device='cpu',
                         feature_cols=['close', 'open', 'high', 'low', 'volume',
                                       'USD_RUB', 'close_RUR', 'RSI_14',
                                       'RSI_14_MA14', 'MACD', 'MACD_Signal',
                                       'MACD_Hist', 'tema', 'macd',
                                       'macd_signal', 'macd_hist', 'SMA_20',
                                       'SMA_50'],
                         max_epochs=25, verbose=0)

In [33]:
y_pred = clf_lstm.predict(df)

## Добавление LSTM модели в ансамбль

In [34]:
# Create a custom ensemble class that doesn't rely on VotingClassifier
class CustomEnsembleClassifier(BaseEstimator, ClassifierMixin):
    """
    Custom ensemble classifier that combines rule-based and ML approaches
    """
    
    def __init__(self, cat_pipeline, rule_pipeline, lstm_pipeline, weights=[0.33, 0.33, 0.34]):
        self.cat_pipeline = cat_pipeline
        self.rule_pipeline = rule_pipeline
        self.lstm_pipeline = lstm_pipeline
        self.weights = weights
        
    def fit(self, X, y=None):
        # Fit both pipelines
        self.cat_pipeline.fit(X, y)
        self.rule_pipeline.fit(X, y)
        self.lstm_pipeline.fit(X, y)
        
        # Set classes
        self.classes_ = np.array([-1, 0, 1])
        
        return self
    
    def predict_proba(self, X):
        # Get probabilities from both models
        cat_proba = self.cat_pipeline.predict_proba(X)
        rule_proba = self.rule_pipeline.predict_proba(X)
        lstm_proba = self.lstm_pipeline.predict_proba(X)
        
        # Weighted average
        ensemble_proba = (self.weights[0] * cat_proba + 
                         self.weights[1] * rule_proba +
                         self.weights[2] * lstm_proba)
        
        return ensemble_proba
    
    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]

In [35]:
# Create custom ensemble
ensemble = CustomEnsembleClassifier(
    cat_pipeline=pipe_cat,
    rule_pipeline=pipe_rule,
    lstm_pipeline = clf_lstm,
    )

ensemble

CustomEnsembleClassifier(cat_pipeline=Pipeline(steps=[('tech',
                                                       TechIndicatorsTransformer()),
                                                      ('scaler',
                                                       DataFrameMinMaxScaler()),
                                                      ('cat',
                                                       CatBoostSignalClassifier(cat_params={'depth': 6,
                                                                                            'iterations': 500,
                                                                                            'learning_rate': 0.05,
                                                                                            'loss_function': 'MultiClass',
                                                                                            'random_seed': 42,
                                                                                            'verbose': False},
                                                                                const_margin=0.001))]),
                         lstm_pipeline=TimeSeriesLSTMClassifier(device='cpu',
                                                                feature_cols=['close',
                                                                              'open',
                                                                              'high',
                                                                              'low',
                                                                              'volume',
                                                                              'USD_RUB',
                                                                              'close_RUR',
                                                                              'RSI_14',
                                                                              'RSI_14_MA14',
                                                                              'MACD',
                                                                              'MACD_Signal',
                                                                              'MACD_Hist',
                                                                              'tema',
                                                                              'macd',
                                                                              'macd_signal',
                                                                              'macd_hist',
                                                                              'SMA_20',
                                                                              'SMA_50'],
                                                                max_epochs=25,
                                                                verbose=0),
                         rule_pipeline=Pipeline(steps=[('tech',
                                                        TechIndicatorsTransformer()),
                                                       ('rule',
                                                        RuleSignalClassifier())]))

In [36]:
ensemble.fit(df, df['target'])

CustomEnsembleClassifier(cat_pipeline=Pipeline(steps=[('tech',
                                                       TechIndicatorsTransformer()),
                                                      ('scaler',
                                                       DataFrameMinMaxScaler()),
                                                      ('cat',
                                                       CatBoostSignalClassifier(cat_params={'depth': 6,
                                                                                            'iterations': 500,
                                                                                            'learning_rate': 0.05,
                                                                                            'loss_function': 'MultiClass',
                                                                                            'random_seed': 42,
                                                                                            'verbose': False},
                                                                                const_margin=0.001))]),
                         lstm_pipeline=TimeSeriesLSTMClassifier(device='cpu',
                                                                feature_cols=['close',
                                                                              'open',
                                                                              'high',
                                                                              'low',
                                                                              'volume',
                                                                              'USD_RUB',
                                                                              'close_RUR',
                                                                              'RSI_14',
                                                                              'RSI_14_MA14',
                                                                              'MACD',
                                                                              'MACD_Signal',
                                                                              'MACD_Hist',
                                                                              'tema',
                                                                              'macd',
                                                                              'macd_signal',
                                                                              'macd_hist',
                                                                              'SMA_20',
                                                                              'SMA_50'],
                                                                max_epochs=25,
                                                                verbose=0),
                         rule_pipeline=Pipeline(steps=[('tech',
                                                        TechIndicatorsTransformer()),
                                                       ('rule',
                                                        RuleSignalClassifier())]))

<div class="alert alert-warning">

<b>Плохо!</b>

Наш класс принимает фиксированное число estimator'ов
    
</div>

Давайте сделаем его универсальным

In [37]:
class CustomEnsembleClassifier(BaseEstimator, ClassifierMixin):
    """
    Custom ensemble classifier that combines any number of classifiers with weights.
    Each classifier must implement fit, predict_proba (как в sklearn).
    """

    def __init__(self, classifiers, weights=None, classes=None):
        """
        Parameters
        ----------
        classifiers : list
            Список классификаторов (sklearn-style объектов).
        weights : list or None
            Веса для классификаторов (нормализуются).
            Если None -> равномерные.
        classes : list or None
            Классы (например [-1, 0, 1]).
            Если None -> берём из первого классификатора после fit.
        """
        self.classifiers = classifiers
        self.weights = weights
        self.classes = classes

    def fit(self, X, y=None):
        # обучаем каждый классификатор (делаем clone чтобы избежать переобучения одних и тех же инстансов)
        self.fitted_classifiers_ = []
        for clf in self.classifiers:
            fitted = clone(clf)
            fitted.fit(X, y)
            self.fitted_classifiers_.append(fitted)

        # классы
        if self.classes is None:
            self.classes_ = np.array(self.fitted_classifiers_[0].classes_)
        else:
            self.classes_ = np.array(self.classes)

        # веса
        if self.weights is None:
            self.weights_ = np.ones(len(self.fitted_classifiers_)) / len(self.fitted_classifiers_)
        else:
            w = np.asarray(self.weights, dtype=float)
            self.weights_ = w / w.sum()

        return self

    def predict_proba(self, X):
        # собираем вероятности от всех классификаторов
        probas = []
        for clf, w in zip(self.fitted_classifiers_, self.weights_):
            p = clf.predict_proba(X)
            probas.append(w * p)
        return np.sum(probas, axis=0)

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]

In [38]:
ensemble = CustomEnsembleClassifier(
    classifiers=[
        ('categories', pipe_cat), 
        ('rule', pipe_rule), 
        ('lstm', clf_lstm)
        ],
    weights=[0.3, 0.2, 0.5],    # если не задать -> равномерные
    classes=[-1, 0, 1]          # если не задать, то берём из первого классификатора после fit.
)

ensemble

CustomEnsembleClassifier(classes=[-1, 0, 1],
                         classifiers=[('categories',
                                       Pipeline(steps=[('tech',
                                                        TechIndicatorsTransformer()),
                                                       ('scaler',
                                                        DataFrameMinMaxScaler()),
                                                       ('cat',
                                                        CatBoostSignalClassifier(cat_params={'depth': 6,
                                                                                             'iterations': 500,
                                                                                             'learning_rate': 0.05,
                                                                                             'loss_function': 'MultiClass',
                                                                                             'random_seed': 42,
                                                                                             'verbose': False},
                                                                                 const_margin=0.001))])),
                                      ('rule',
                                       Pipeli...h',
                                                        TechIndicatorsTransformer()),
                                                       ('rule',
                                                        RuleSignalClassifier())])),
                                      ('lstm',
                                       TimeSeriesLSTMClassifier(device='cpu',
                                                                feature_cols=['close',
                                                                              'open',
                                                                              'high',
                                                                              'low',
                                                                              'volume',
                                                                              'USD_RUB',
                                                                              'close_RUR',
                                                                              'RSI_14',
                                                                              'RSI_14_MA14',
                                                                              'MACD',
                                                                              'MACD_Signal',
                                                                              'MACD_Hist',
                                                                              'tema',
                                                                              'macd',
                                                                              'macd_signal',
                                                                              'macd_hist',
                                                                              'SMA_20',
                                                                              'SMA_50'],
                                                                max_epochs=25,
                                                                verbose=0))],
                         weights=[0.3, 0.2, 0.5])

In [39]:
ensemble.fit(df, df['target'])

TypeError: Cannot clone object ''categories'' (type <class 'str'>): it does not seem to be a scikit-learn estimator as it does not implement a 'get_params' method.

<div class="alert alert-warning">

<b>Ошибка</b>

Ошибка приходит из sklearn.clone: внутри какого-то из наших пайплайнов/классификаторов есть параметр типа строка (например 'categories'), и clone пытается рекурсивно «клонировать» элементы списков/кортежей и натыкается на строку → TypeError.
    
</div>

In [40]:
class CustomEnsembleClassifier(BaseEstimator, ClassifierMixin):
    """
    Универсальный ансамбль произвольного числа классификаторов.
    Требует наличия у подмоделей методов fit(X,y) и predict_proba(X).
    """

    def __init__(self, classifiers, weights=None, classes=None, deepcopy_estimators=False):
        """
        Parameters
        ----------
        classifiers : list
            Список базовых моделей (sklearn-совместимых).
        weights : list | None
            Веса моделей (нормализуются). По умолчанию равномерные.
        classes : array-like | None
            Фиксированный порядок классов ансамбля. Если None — берём объединение
            всех classes_ после fit.
        deepcopy_estimators : bool
            Если True — делаем deepcopy(estimator) перед fit, иначе обучаем переданные инстансы.
        """
        self.classifiers = classifiers
        self.weights = weights
        self.classes = classes
        self.deepcopy_estimators = deepcopy_estimators

    # ---------- priv helpers ----------
    @staticmethod
    def _ensure_2d(a):
        a = np.asarray(a)
        return a if a.ndim == 2 else a.reshape(-1, 1)

    # ---------- sklearn API ----------
    def fit(self, X, y=None):
        if not isinstance(self.classifiers, (list, tuple)) or len(self.classifiers) == 0:
            raise ValueError("Передайте непустой список классификаторов.")

        # 1) обучаем базовые модели
        self.fitted_ = []
        for est in self.classifiers:
            if est is None:
                raise ValueError("Обнаружен None в списке классификаторов.")
            fitted = deepcopy(est) if self.deepcopy_estimators else est
            if not hasattr(fitted, "fit") or not hasattr(fitted, "predict_proba"):
                raise TypeError(f"Классификатор {type(fitted)} должен иметь fit и predict_proba.")
            fitted.fit(X, y)
            self.fitted_.append(fitted)

        # 2) общий список классов ансамбля
        if self.classes is not None:
            self.classes_ = np.asarray(self.classes)
        else:
            all_classes = []
            for est in self.fitted_:
                if not hasattr(est, "classes_"):
                    raise AttributeError(f"{type(est)} не имеет атрибута classes_ после fit.")
                all_classes.append(np.asarray(est.classes_))
            self.classes_ = np.unique(np.concatenate(all_classes))

        # 3) веса
        n = len(self.fitted_)
        if self.weights is None:
            self.weights_ = np.ones(n, dtype=float) / n
        else:
            w = np.asarray(self.weights, dtype=float)
            if w.shape[0] != n:
                raise ValueError(f"Длина weights ({w.shape[0]}) != числу классификаторов ({n}).")
            s = w.sum()
            if s <= 0:
                raise ValueError("Сумма весов должна быть > 0.")
            self.weights_ = w / s

        return self

    def _model_proba_aligned(self, est, X):
        """
        Возвращает вероятности, выровненные по self.classes_.
        """
        proba = est.predict_proba(X)  # (N, C_est)
        est_classes = np.asarray(est.classes_)
        N = proba.shape[0]
        C = len(self.classes_)
        out = np.zeros((N, C), dtype=float)
        # разместим столбцы в соответствии с индексами
        for j, cls in enumerate(est_classes):
            idx = np.where(self.classes_ == cls)[0]
            if idx.size:
                out[:, idx[0]] = proba[:, j]
            # если у ансамбля есть класс, которого нет у модели — он останется с нулевой вероятностью
        return out

    def predict_proba(self, X):
        check_is_fitted(self, ["fitted_", "classes_", "weights_"])
        accum = None
        for est, w in zip(self.fitted_, self.weights_):
            p = self._model_proba_aligned(est, X)
            accum = p * w if accum is None else accum + p * w
        return accum

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]

In [41]:
ensemble = CustomEnsembleClassifier(
    classifiers=[pipe_cat, pipe_rule, clf_lstm],         # сколько угодно estimator'ов
    weights=[0.33, 0.33, 0.34],                          # или None для равных
    classes=[-1, 0, 1],                                  # фиксируем порядок классов
    deepcopy_estimators=False                            # можно True, если хотите не трогать исходные инстансы
)

In [42]:
ensemble.fit(df, df['target'])

CustomEnsembleClassifier(classes=[-1, 0, 1],
                         classifiers=[Pipeline(steps=[('tech',
                                                       TechIndicatorsTransformer()),
                                                      ('scaler',
                                                       DataFrameMinMaxScaler()),
                                                      ('cat',
                                                       CatBoostSignalClassifier(cat_params={'depth': 6,
                                                                                            'iterations': 500,
                                                                                            'learning_rate': 0.05,
                                                                                            'loss_function': 'MultiClass',
                                                                                            'random_seed': 42,
                                                                                            'verbose': False},
                                                                                const_margin=0.001))]),
                                      Pipeline(steps=[('tech',
                                                       TechIndicatorsTransformer()),
                                                      ('rule',
                                                       RuleSignalClassifier())]),
                                      TimeSeriesLSTMClassifier(device='cpu',
                                                               feature_cols=['close',
                                                                             'open',
                                                                             'high',
                                                                             'low',
                                                                             'volume',
                                                                             'USD_RUB',
                                                                             'close_RUR',
                                                                             'RSI_14',
                                                                             'RSI_14_MA14',
                                                                             'MACD',
                                                                             'MACD_Signal',
                                                                             'MACD_Hist',
                                                                             'tema',
                                                                             'macd',
                                                                             'macd_signal',
                                                                             'macd_hist',
                                                                             'SMA_20',
                                                                             'SMA_50'],
                                                               max_epochs=25,
                                                               verbose=0)],
                         weights=[0.33, 0.33, 0.34])

In [43]:
y_pred = ensemble.predict(df)

ValueError: operands could not be broadcast together with shapes (3849,3) (3848,3) 

<div class="alert alert-warning">

<b>Ошибка</b>

Разная длина выходов predict_proba у базовых моделей. Для тайм-серий это частая история: где-то внутри пайплайна отрезались первые или последние строки
    
</div>

In [44]:
class CustomEnsembleClassifier(BaseEstimator, ClassifierMixin):
    """
    Универсальный ансамбль произвольного числа классификаторов.
    Поддерживает несоответствие длин predict_proba у базовых моделей: выравнивает по left/right.
    """

    def __init__(self, classifiers, weights=None, classes=None,
                 deepcopy_estimators=False, align_mode="right"):
        """
        classifiers : list      — базовые модели (fit, predict_proba обязательны)
        weights     : list|None — веса (нормализуются), по умолчанию равные
        classes     : list|None — порядок классов ансамбля; если None — объединение всех
        deepcopy_estimators : bool — обучать deepcopy(estimator), иначе инстансы как есть
        align_mode  : {'right','left'} — как выравнивать длины: последние или первые min_len строк
        """
        self.classifiers = classifiers
        self.weights = weights
        self.classes = classes
        self.deepcopy_estimators = deepcopy_estimators
        self.align_mode = align_mode

    def fit(self, X, y=None):
        if not isinstance(self.classifiers, (list, tuple)) or len(self.classifiers) == 0:
            raise ValueError("Передайте непустой список классификаторов.")
        if self.align_mode not in ("right", "left"):
            raise ValueError("align_mode должен быть 'right' или 'left'.")

        self.fitted_ = []
        for est in self.classifiers:
            if est is None:
                raise ValueError("Обнаружен None в списке классификаторов.")
            fitted = deepcopy(est) if self.deepcopy_estimators else est
            if not hasattr(fitted, "fit") or not hasattr(fitted, "predict_proba"):
                raise TypeError(f"{type(fitted)} должен иметь методы fit и predict_proba.")
            fitted.fit(X, y)
            self.fitted_.append(fitted)

        # общий набор классов
        if self.classes is not None:
            self.classes_ = np.asarray(self.classes)
        else:
            all_classes = []
            for est in self.fitted_:
                if not hasattr(est, "classes_"):
                    raise AttributeError(f"{type(est)} не имеет classes_ после fit.")
                all_classes.append(np.asarray(est.classes_))
            self.classes_ = np.unique(np.concatenate(all_classes))

        # веса
        n = len(self.fitted_)
        if self.weights is None:
            self.weights_ = np.ones(n, dtype=float) / n
        else:
            w = np.asarray(self.weights, dtype=float)
            if w.shape[0] != n:
                raise ValueError(f"Длина weights ({w.shape[0]}) != числу моделей ({n}).")
            s = w.sum()
            if s <= 0:
                raise ValueError("Сумма весов должна быть > 0.")
            self.weights_ = w / s

        return self

    def _align_classes(self, proba, est_classes):
        """Проецирует вероятности модели на порядок self.classes_."""
        est_classes = np.asarray(est_classes)
        N, C_all = proba.shape[0], len(self.classes_)
        out = np.zeros((N, C_all), dtype=float)
        for j, cls in enumerate(est_classes):
            idx = np.where(self.classes_ == cls)[0]
            if idx.size:
                out[:, idx[0]] = proba[:, j]
        return out

    def _collect_and_align(self, X):
        """Собирает proba всех моделей, выравнивает по классам и по длине."""
        aligned = []
        lengths = []
        for est in self.fitted_:
            p = est.predict_proba(X)
            p = np.asarray(p)
            if p.ndim != 2:
                raise ValueError(f"{type(est)}.predict_proba вернул не (N,C).")
            p = self._align_classes(p, est.classes_)
            aligned.append(p)
            lengths.append(p.shape[0])

        # выравнивание по длине
        min_len = min(lengths)
        if len(set(lengths)) > 1:
            warnings.warn(
                f"Разная длина predict_proba у базовых моделей {lengths}; "
                f"выравниваю по {self.align_mode}, итог N={min_len}."
            )
        if self.align_mode == "right":
            aligned = [p[-min_len:, :] for p in aligned]
        else:  # 'left'
            aligned = [p[:min_len, :] for p in aligned]

        return aligned, min_len

    def predict_proba(self, X):
        check_is_fitted(self, ["fitted_", "classes_", "weights_"])
        probas, _ = self._collect_and_align(X)
        # взвешенная сумма
        accum = None
        for p, w in zip(probas, self.weights_):
            accum = p * w if accum is None else accum + p * w
        return accum

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]

In [45]:
ensemble = CustomEnsembleClassifier(
    classifiers=[pipe_cat, pipe_rule, clf_lstm],         # сколько угодно estimator'ов
    weights=[0.33, 0.33, 0.34],                          # или None для равных
    classes=[-1, 0, 1],                                  # фиксируем порядок классов
    deepcopy_estimators=False                            # можно True, если хотите не трогать исходные инстансы
)

In [46]:
ensemble.fit(df, df['target'])

CustomEnsembleClassifier(classes=[-1, 0, 1],
                         classifiers=[Pipeline(steps=[('tech',
                                                       TechIndicatorsTransformer()),
                                                      ('scaler',
                                                       DataFrameMinMaxScaler()),
                                                      ('cat',
                                                       CatBoostSignalClassifier(cat_params={'depth': 6,
                                                                                            'iterations': 500,
                                                                                            'learning_rate': 0.05,
                                                                                            'loss_function': 'MultiClass',
                                                                                            'random_seed': 42,
                                                                                            'verbose': False},
                                                                                const_margin=0.001))]),
                                      Pipeline(steps=[('tech',
                                                       TechIndicatorsTransformer()),
                                                      ('rule',
                                                       RuleSignalClassifier())]),
                                      TimeSeriesLSTMClassifier(device='cpu',
                                                               feature_cols=['close',
                                                                             'open',
                                                                             'high',
                                                                             'low',
                                                                             'volume',
                                                                             'USD_RUB',
                                                                             'close_RUR',
                                                                             'RSI_14',
                                                                             'RSI_14_MA14',
                                                                             'MACD',
                                                                             'MACD_Signal',
                                                                             'MACD_Hist',
                                                                             'tema',
                                                                             'macd',
                                                                             'macd_signal',
                                                                             'macd_hist',
                                                                             'SMA_20',
                                                                             'SMA_50'],
                                                               max_epochs=25,
                                                               verbose=0)],
                         weights=[0.33, 0.33, 0.34])

In [47]:
y_pred = ensemble.predict(df)
y_proba = ensemble.predict_proba(df)

<div class="alert alert-success">

<b>Отлично ✅</b>

Мы собрали кастомный анасмбль на интерфейсе sklearn для произвольного числа estimator'ов
    
</div>

## Добавим модель которая "балансирует" веса - вариант Mixture-of-Experts с гейтинг-сетью

Соберем адаптивный MoE-ансамбль (mixture-of-experts) для произвольного числа базовых моделей. Он обучает гейтинг-модель, которая по признакам (и/или по прогнозам базовых) выдаёт веса $w_k(x)$ для каждого эксперта; итоговая вероятность — $\sum_k w_k(x)\,p_k(y\mid x)$. 
В тренировке используем OOF-предсказания базовых, чтобы избежать утечки.
- Поддерживает: любой список классификаторов с `fit/predict_proba`, разные наборы `classes_`, «правое/левое» выравнивание длин.
- Веса меняются по объекту.
- Есть метод `predict_weights(X)` — увидеть, кто «рулит» где.

In [48]:
class _GatingNet(nn.Module):
    """
    Небольшая MLP, которая по фичам G(x) выдаёт логиты весов экспертов.
    Softmax(logits) -> w_k(x).
    """
    def __init__(self, in_dim, n_experts, hidden=64, depth=2, dropout=0.0):
        super().__init__()
        layers = []
        h = in_dim
        for _ in range(depth):
            layers += [nn.Linear(h, hidden), nn.ReLU()]
            if dropout > 0:
                layers += [nn.Dropout(dropout)]
            h = hidden
        layers += [nn.Linear(h, n_experts)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)  # (B, n_experts)

In [ ]:
class AdaptiveMixtureEnsemble(BaseEstimator, ClassifierMixin):
    """
    Mixture-of-Experts ансамбль произвольного числа классификаторов.
    Базовые модели дают p_k(y|x), гейтинг-сеть учит веса w_k(x) по признакам и/или proba базовых.
    Итог: p_mix(y|x) = sum_k w_k(x) * p_k(y|x).

    Параметры:
    ----------
    classifiers : list
        Любые модели с fit/predict_proba.
    classes : list|None
        Порядок классов ансамбля; None -> объединение всех классов базовых.
    deepcopy_estimators : bool
        Делать deepcopy при обучении (без него — фитим переданные инстансы).
    align_mode : {'right','left'}
        Как выравнивать длину, если модели выдают разное N (например, окна/лаговые пайплайны).

    use_X : bool
        Включать ли X в фичи для гейтинга.
    use_base_proba : bool
        Включать ли (OOF) вероятности базовых в фичи для гейтинга.
    extra_features : {'none','entropy','confidence','all'}
        Добавить ли агрегаты по базовым (энтропия, максимум вероятности и т.п.).
    cv : int
        Кол-во фолдов для OOF.

    gate_hidden, gate_depth, gate_dropout
        Архитектура MLP гейтинга.
    epochs, lr, weight_decay, batch_size, device
        Гиперпараметры обучения гейтинга.

    random_state : int
        Сид для KFold (перемешивание).
    """

    def __init__(self,
                 classifiers,
                 classes=None,
                 deepcopy_estimators=False,
                 align_mode="right",

                 use_X=True,
                 use_base_proba=True,
                 extra_features="none",  # 'none'|'entropy'|'confidence'|'all'
                 cv=5,

                 gate_hidden=64,
                 gate_depth=2,
                 gate_dropout=0.0,

                 epochs=50,
                 lr=1e-3,
                 weight_decay=1e-4,
                 batch_size=2048,
                 device='cpu',

                 random_state=42):
        self.classifiers = classifiers
        self.classes = classes
        self.deepcopy_estimators = deepcopy_estimators
        self.align_mode = align_mode

        self.use_X = use_X
        self.use_base_proba = use_base_proba
        self.extra_features = extra_features
        self.cv = cv

        self.gate_hidden = gate_hidden
        self.gate_depth = gate_depth
        self.gate_dropout = gate_dropout

        self.epochs = epochs
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.device = device

        self.random_state = random_state

    # ---------- служебные ----------
    def _check_inputs(self):
        if not isinstance(self.classifiers, (list, tuple)) or len(self.classifiers) == 0:
            raise ValueError("Передайте непустой список классификаторов.")
        if self.align_mode not in ("right", "left"):
            raise ValueError("align_mode должен быть 'right' или 'left'.")

    def _infer_classes_union(self, fitted_list):
        if self.classes is not None:
            return np.asarray(self.classes)
        all_classes = []
        for est in fitted_list:
            if not hasattr(est, "classes_"):
                raise AttributeError(f"{type(est)} не имеет classes_ после fit.")
            all_classes.append(np.asarray(est.classes_))
        return np.unique(np.concatenate(all_classes))

    @staticmethod
    def _align_one(proba, est_classes, target_classes):
        """Проецирует proba модели на target_classes (порядок ансамбля)."""
        est_classes = np.asarray(est_classes)
        N, C_all = proba.shape[0], len(target_classes)
        out = np.zeros((N, C_all), dtype=float)
        # переносим столбцы по совпадающим классам
        for j, cls in enumerate(est_classes):
            idx = np.where(target_classes == cls)[0]
            if idx.size:
                out[:, idx[0]] = proba[:, j]
        return out

    def _collect_and_align(self, X, fitted_list, classes_order):
        """Собирает proba всех моделей, выравнивает по классам и по длине."""
        aligned = []
        lengths = []
        for est in fitted_list:
            p = np.asarray(est.predict_proba(X))
            if p.ndim != 2:
                raise ValueError(f"{type(est)}.predict_proba вернул не (N,C).")
            p = self._align_one(p, est.classes_, classes_order)
            aligned.append(p)
            lengths.append(p.shape[0])

        min_len = min(lengths)
        if len(set(lengths)) > 1:
            warnings.warn(
                f"Разная длина predict_proba у базовых моделей {lengths}; "
                f"выравниваю по {self.align_mode}, итог N={min_len}."
            )
        if self.align_mode == "right":
            aligned = [p[-min_len:, :] for p in aligned]
        else:
            aligned = [p[:min_len, :] for p in aligned]
        return aligned, min_len

    def _oof_proba_all(self, X, y, est):
        """OOF-предсказания одного базового классификатора, выровненные по self.classes_."""
        kf = KFold(n_splits=self.cv, shuffle=True, random_state=self.random_state)
        n_classes = len(self.classes_)
        oof = np.zeros((X.shape[0], n_classes), dtype=float)

        for tr, te in kf.split(X):
            est_fold = clone(est)
            est_fold.fit(X[tr], y[tr])
            p = est_fold.predict_proba(X[te])
            p_aligned = self._align_one(p, est_fold.classes_, self.classes_)
            oof[te] = p_aligned
        return oof

    @staticmethod
    def _entropy(p, eps=1e-12):
        p = np.clip(p, eps, 1.0)
        return -(p * np.log(p)).sum(axis=1, keepdims=True)

    @staticmethod
    def _confidence(p):
        return p.max(axis=1, keepdims=True)

    def _build_gate_features(self, X, stacked_prob_list):
        """
        Собираем фичи для гейтинга:
        - X (опционально)
        - вектор p_k(y|x) всех экспертов, конкатенация [K * C]
        - агрегаты (энтропия/конфиденс) по каждому эксперту (если нужно)
        """
        feats = []
        if self.use_X:
            feats.append(X)

        if self.use_base_proba:
            # [n_samples, K*C]
            feats.append(np.hstack(stacked_prob_list))

            if self.extra_features in ("entropy", "all"):
                ents = [self._entropy(p) for p in stacked_prob_list]
                feats.append(np.hstack(ents))  # [n_samples, K]

            if self.extra_features in ("confidence", "all"):
                confs = [self._confidence(p) for p in stacked_prob_list]
                feats.append(np.hstack(confs))  # [n_samples, K]

        if not feats:
            # Минимальный вариант — хотя бы proba первого эксперта
            feats.append(stacked_prob_list[0])

        return np.hstack(feats)

    # ---------- основной ----------
    def fit(self, X, y):
        self._check_inputs()
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y)

        # 1) Фитим базовые на всех данных (или deepcopy)
        self.fitted_ = []
        for est in self.classifiers:
            if est is None:
                raise ValueError("Обнаружен None в списке классификаторов.")
            fitted = deepcopy(est) if self.deepcopy_estimators else est
            if not hasattr(fitted, "fit") or not hasattr(fitted, "predict_proba"):
                raise TypeError(f"{type(fitted)} должен иметь методы fit и predict_proba.")
            fitted.fit(X, y)
            self.fitted_.append(fitted)

        # 2) Определяем общий порядок классов
        self.classes_ = self._infer_classes_union(self.fitted_)

        # 3) OOF для гейтинга (без утечки)
        oof_list = []
        for est in self.classifiers:
            oof = self._oof_proba_all(X, y, est)
            oof_list.append(oof)

        # 4) Собираем фичи для гейтинга
        G = self._build_gate_features(X, oof_list).astype(np.float32)

        # 5) Таргет как индексы классов
        y_idx = np.vectorize(lambda v: np.where(self.classes_ == v)[0][0])(y)

        # 6) Фиксируем OOF-вероятности как «эксперты» для обучения
        base_probs = np.stack(oof_list, axis=1)  # [N, K, C]
        N, K, C = base_probs.shape

        # 7) Создаем и учим гейтинг MLP
        self.n_experts_ = K
        self.gate_ = _GatingNet(
            in_dim=G.shape[1],
            n_experts=K,
            hidden=self.gate_hidden,
            depth=self.gate_depth,
            dropout=self.gate_dropout
        ).to(self.device)

        opt = optim.Adam(self.gate_.parameters(), lr=self.lr, weight_decay=self.weight_decay)

        # В тензоры
        G_t = torch.tensor(G, dtype=torch.float32, device=self.device)
        Y_t = torch.tensor(y_idx, dtype=torch.long, device=self.device)
        B_t = torch.tensor(base_probs, dtype=torch.float32, device=self.device)

        # батч-цикл
        idx_all = np.arange(N)
        for _ in range(self.epochs):
            np.random.shuffle(idx_all)
            for i0 in range(0, N, self.batch_size):
                ids = idx_all[i0:i0 + self.batch_size]
                g = G_t[ids]           # (B, in)
                yb = Y_t[ids]          # (B,)
                pb = B_t[ids]          # (B, K, C)

                logits_w = self.gate_(g)              # (B, K)
                w = torch.softmax(logits_w, dim=1)    # (B, K)

                mix = (w.unsqueeze(-1) * pb).sum(dim=1)  # (B, C)
                p_true = mix[torch.arange(mix.size(0)), yb]
                loss = -torch.log(p_true.clamp_min(1e-12)).mean()

                opt.zero_grad()
                loss.backward()
                opt.step()

        return self

    def _current_base_prob_list(self, X):
        """
        Текущие вероятности экспертов от фулл-фитов, выровненные и подрезанные по длине.
        """
        aligned_list, min_len = self._collect_and_align(X, self.fitted_, self.classes_)
        return aligned_list, min_len

    def predict_proba(self, X):
        check_is_fitted(self, ["fitted_", "classes_", "gate_", "n_experts_"])
        X = np.asarray(X, dtype=np.float32)

        # 1) proba экспертов на инференсе
        prob_list, n = self._current_base_prob_list(X)  # список длины K, каждый [n, C]
        K = len(prob_list)
        C = prob_list[0].shape[1]

        # 2) фичи гейтинга на инференсе (можно заменить OOF на текущие proba)
        G = self._build_gate_features(X[-n:], prob_list).astype(np.float32)

        # 3) веса w(x)
        self.gate_.eval()
        with torch.no_grad():
            G_t = torch.tensor(G, dtype=torch.float32, device=self.device)
            logits_w = self.gate_(G_t)              # (n, K)
            w = torch.softmax(logits_w, dim=1).cpu().numpy()  # (n, K)

        # 4) смесь
        mix = np.zeros((n, C), dtype=float)
        for k in range(K):
            mix += w[:, [k]] * prob_list[k]
        return mix

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]

    def predict_weights(self, X):
        """
        Вернёт матрицу весов w(x): shape = (N_common, K).
        Полезно для анализа вклада экспертов.
        """
        check_is_fitted(self, ["fitted_", "classes_", "gate_", "n_experts_"])
        X = np.asarray(X, dtype=np.float32)
        prob_list, n = self._current_base_prob_list(X)
        G = self._build_gate_features(X[-n:], prob_list).astype(np.float32)

        self.gate_.eval()
        with torch.no_grad():
            G_t = torch.tensor(G, dtype=torch.float32, device=self.device)
            logits_w = self.gate_(G_t)
            w = torch.softmax(logits_w, dim=1).cpu().numpy()
        return w  # (n, K)

In [50]:
ensemble = AdaptiveMixtureEnsemble(
    classifiers=[pipe_cat, pipe_rule, clf_lstm],
    classes=[-1, 0, 1],          # или None -> объединение
    deepcopy_estimators=True,    # чтобы не перезаписывать исходные инстансы

    use_X=True,                  # добавляем исходные признаки в гейтинг
    use_base_proba=True,         # и OOF / proba базовых
    extra_features="all",        # энтропия и максимум как доп. фичи
    cv=5,

    gate_hidden=64,
    gate_depth=2,
    gate_dropout=0.1,

    epochs=60,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=4096,
    device="cpu",                # или "cuda" если доступна
    random_state=42
)

In [51]:
ensemble.fit(df, df['target'])

TypeError: float() argument must be a string or a real number, not 'Timestamp'

<div class="alert alert-warning">

<b>Ошибка</b>

Падает на попытке превратить весь df в float32: у нас есть datetime-колонки (Timestamp), которые так не кастятся. 

Базовые пайплайны пусть получают X «как есть», а в float32 нам нужен только вход в гейтинг-сеть.
    
</div>

In [ ]:
class AdaptiveMixtureEnsemble(BaseEstimator, ClassifierMixin):
    """
    Mixture-of-Experts ансамбль произвольного числа классификаторов.
    Базовые модели дают p_k(y|x), гейтинг-сеть учит веса w_k(x) по признакам и/или proba базовых.
    Итог: p_mix(y|x) = sum_k w_k(x) * p_k(y|x).

    Параметры:
    ----------
    classifiers : list
        Любые модели с fit/predict_proba.
    classes : list|None
        Порядок классов ансамбля; None -> объединение всех классов базовых.
    deepcopy_estimators : bool
        Делать deepcopy при обучении (без него — фитим переданные инстансы).
    align_mode : {'right','left'}
        Как выравнивать длину, если модели выдают разное N (например, окна/лаговые пайплайны).

    use_X : bool
        Включать ли X в фичи для гейтинга.
    use_base_proba : bool
        Включать ли (OOF) вероятности базовых в фичи для гейтинга.
    extra_features : {'none','entropy','confidence','all'}
        Добавить ли агрегаты по базовым (энтропия, максимум вероятности и т.п.).
    cv : int
        Кол-во фолдов для OOF.

    gate_hidden, gate_depth, gate_dropout
        Архитектура MLP гейтинга.
    epochs, lr, weight_decay, batch_size, device
        Гиперпараметры обучения гейтинга.

    random_state : int
        Сид для KFold (перемешивание).
    """

    def __init__(self,
                 classifiers,
                 classes=None,
                 deepcopy_estimators=False,
                 align_mode="right",

                 use_X=True,
                 use_base_proba=True,
                 extra_features="none",  # 'none'|'entropy'|'confidence'|'all'
                 cv=5,

                 gate_hidden=64,
                 gate_depth=2,
                 gate_dropout=0.0,

                 epochs=50,
                 lr=1e-3,
                 weight_decay=1e-4,
                 batch_size=2048,
                 device='cpu',

                 random_state=42):
        self.classifiers = classifiers
        self.classes = classes
        self.deepcopy_estimators = deepcopy_estimators
        self.align_mode = align_mode

        self.use_X = use_X
        self.use_base_proba = use_base_proba
        self.extra_features = extra_features
        self.cv = cv

        self.gate_hidden = gate_hidden
        self.gate_depth = gate_depth
        self.gate_dropout = gate_dropout

        self.epochs = epochs
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.device = device

        self.random_state = random_state

    # ---------- служебные ----------
    def _check_inputs(self):
        if not isinstance(self.classifiers, (list, tuple)) or len(self.classifiers) == 0:
            raise ValueError("Передайте непустой список классификаторов.")
        if self.align_mode not in ("right", "left"):
            raise ValueError("align_mode должен быть 'right' или 'left'.")

    def _infer_classes_union(self, fitted_list):
        if self.classes is not None:
            return np.asarray(self.classes)
        all_classes = []
        for est in fitted_list:
            if not hasattr(est, "classes_"):
                raise AttributeError(f"{type(est)} не имеет classes_ после fit.")
            all_classes.append(np.asarray(est.classes_))
        return np.unique(np.concatenate(all_classes))
    
    def _numerify_X(self, X):
        """
        Превращает X (pd.DataFrame/ndarray) в np.float32 2D без падений:
        - datetime64 -> int64 наносекунды (можно делить на 1e9, если хочется секунды)
        - bool -> {0,1}
        - category -> .cat.codes
        - object -> pd.to_numeric(errors='coerce')
        """
        import pandas as pd
        if isinstance(X, pd.DataFrame):
            X_num = X.copy()
            for col in X_num.columns:
                dt = X_num[col].dtype
                if pd.api.types.is_datetime64_any_dtype(dt):
                    X_num[col] = X_num[col].view('int64')  # наносекунды
                elif pd.api.types.is_bool_dtype(dt):
                    X_num[col] = X_num[col].astype('int8')
                elif pd.api.types.is_categorical_dtype(dt):
                    X_num[col] = X_num[col].cat.codes.astype('int32')
                elif pd.api.types.is_object_dtype(dt):
                    X_num[col] = pd.to_numeric(X_num[col], errors='coerce')
                # остальные (int/float) оставляем как есть
            X_num = X_num.fillna(0.0).to_numpy(dtype=np.float32, copy=False)
            return X_num
        else:
            arr = np.asarray(X)
            # если не числовой — пробуем привести
            if not np.issubdtype(arr.dtype, np.number):
                import pandas as pd
                return self._numerify_X(pd.DataFrame(arr))
            return arr.astype(np.float32, copy=False)

    @staticmethod
    def _align_one(proba, est_classes, target_classes):
        """Проецирует proba модели на target_classes (порядок ансамбля)."""
        est_classes = np.asarray(est_classes)
        N, C_all = proba.shape[0], len(target_classes)
        out = np.zeros((N, C_all), dtype=float)
        # переносим столбцы по совпадающим классам
        for j, cls in enumerate(est_classes):
            idx = np.where(target_classes == cls)[0]
            if idx.size:
                out[:, idx[0]] = proba[:, j]
        return out

    def _collect_and_align(self, X, fitted_list, classes_order):
        """Собирает proba всех моделей, выравнивает по классам и по длине."""
        aligned = []
        lengths = []
        for est in fitted_list:
            p = np.asarray(est.predict_proba(X))
            if p.ndim != 2:
                raise ValueError(f"{type(est)}.predict_proba вернул не (N,C).")
            p = self._align_one(p, est.classes_, classes_order)
            aligned.append(p)
            lengths.append(p.shape[0])

        min_len = min(lengths)
        if len(set(lengths)) > 1:
            warnings.warn(
                f"Разная длина predict_proba у базовых моделей {lengths}; "
                f"выравниваю по {self.align_mode}, итог N={min_len}."
            )
        if self.align_mode == "right":
            aligned = [p[-min_len:, :] for p in aligned]
        else:
            aligned = [p[:min_len, :] for p in aligned]
        return aligned, min_len

    def _oof_proba_all(self, X, y, est):
        """OOF-предсказания одного базового классификатора, выровненные по self.classes_."""
        kf = KFold(n_splits=self.cv, shuffle=False)
        n_classes = len(self.classes_)
        oof = np.zeros((X.shape[0], n_classes), dtype=float)
    
        # ВАЖНО: если X — DataFrame, нужно .iloc
        is_df = hasattr(X, "iloc")
        for tr, te in kf.split(np.arange(len(X))):
            # clone → deepcopy fallback
            try:
                est_fold = clone(est)
            except Exception as e:
                warnings.warn(f"Cannot clone {type(est).__name__}: {e}. Fallback to deepcopy.")
                est_fold = deepcopy(est)
                
            X_tr = X.iloc[tr] if is_df else X[tr]
            X_te = X.iloc[te] if is_df else X[te]
            y_tr = y[tr]

            est_fold.fit(X_tr, y_tr)
            p = est_fold.predict_proba(X_te)
            p_aligned = self._align_one(p, est_fold.classes_, self.classes_)
            
            
            # ---- ВЫРАВНИВАНИЕ ДЛИН ДЛЯ OOF ----
            L = len(te)
            m = p_aligned.shape[0]

            if m == L:
                te_use = te
                p_use = p_aligned
            elif m < L:
                # модель вернула меньше строк, чем объектов в fold-е
                if self.align_mode == "right":
                    te_use = te[-m:]
                    p_use = p_aligned
                else:  # 'left'
                    te_use = te[:m]
                    p_use = p_aligned
                warnings.warn(f"{type(est).__name__}: OOF length {m} < {L}, align={self.align_mode}")
            else:  # m > L — на всякий
                if self.align_mode == "right":
                    te_use = te
                    p_use = p_aligned[-L:, :]
                else:
                    te_use = te
                    p_use = p_aligned[:L, :]
                warnings.warn(f"{type(est).__name__}: OOF length {m} > {L}, trimmed with align={self.align_mode}")

            oof[te_use] = p_use
        return oof

    @staticmethod
    def _entropy(p, eps=1e-12):
        p = np.clip(p, eps, 1.0)
        return -(p * np.log(p)).sum(axis=1, keepdims=True)

    @staticmethod
    def _confidence(p):
        return p.max(axis=1, keepdims=True)

    def _build_gate_features(self, X, stacked_prob_list):
        """
        Собираем фичи для гейтинга:
        - X (опционально)
        - вектор p_k(y|x) всех экспертов, конкатенация [K * C]
        - агрегаты (энтропия/конфиденс) по каждому эксперту (если нужно)
        """
        feats = []
        if self.use_X and X is not None:
            feats.append(X)  # уже np.float32
        if self.use_base_proba:
            feats.append(np.hstack(stacked_prob_list))
            if self.extra_features in ("entropy", "all"):
                feats.append(np.hstack([self._entropy(p) for p in stacked_prob_list]))
            if self.extra_features in ("confidence", "all"):
                feats.append(np.hstack([self._confidence(p) for p in stacked_prob_list]))
        if not feats:
            feats.append(stacked_prob_list[0])
        return np.hstack(feats)

    # ---------- основной ----------
    def fit(self, X, y):
        self._check_inputs()
        y = np.asarray(y)

        # 1) Фитим базовые на всех данных (или deepcopy)
        self.fitted_ = []
        for est in self.classifiers:
            if est is None:
                raise ValueError("Обнаружен None в списке классификаторов.")
            fitted = deepcopy(est) if self.deepcopy_estimators else est
            if not hasattr(fitted, "fit") or not hasattr(fitted, "predict_proba"):
                raise TypeError(f"{type(fitted)} должен иметь методы fit и predict_proba.")
            fitted.fit(X, y)
            self.fitted_.append(fitted)

        # 2) Определяем общий порядок классов
        self.classes_ = self._infer_classes_union(self.fitted_)

        # 3) OOF для гейтинга (без утечки)
        oof_list = []
        for est in self.classifiers:
            oof = self._oof_proba_all(X, y, est)
            oof_list.append(oof)

        # 4) Собираем фичи для гейтинга
        X_num = self._numerify_X(X) if self.use_X else None
        G = self._build_gate_features(X_num if X_num is not None else X, oof_list).astype(np.float32)

        # 5) Таргет как индексы классов
        y_idx = np.vectorize(lambda v: np.where(self.classes_ == v)[0][0])(y)

        # 6) Фиксируем OOF-вероятности как «эксперты» для обучения
        base_probs = np.stack(oof_list, axis=1)  # [N, K, C]
        N, K, C = base_probs.shape

        # 7) Создаем и учим гейтинг MLP
        self.n_experts_ = K
        self.gate_ = _GatingNet(
            in_dim=G.shape[1],
            n_experts=K,
            hidden=self.gate_hidden,
            depth=self.gate_depth,
            dropout=self.gate_dropout
        ).to(self.device)

        opt = optim.Adam(self.gate_.parameters(), lr=self.lr, weight_decay=self.weight_decay)

        # В тензоры
        G_t = torch.tensor(G, dtype=torch.float32, device=self.device)
        Y_t = torch.tensor(y_idx, dtype=torch.long, device=self.device)
        B_t = torch.tensor(base_probs, dtype=torch.float32, device=self.device)

        # батч-цикл
        idx_all = np.arange(N)
        for _ in range(self.epochs):
            np.random.shuffle(idx_all)
            for i0 in range(0, N, self.batch_size):
                ids = idx_all[i0:i0 + self.batch_size]
                g = G_t[ids]           # (B, in)
                yb = Y_t[ids]          # (B,)
                pb = B_t[ids]          # (B, K, C)

                logits_w = self.gate_(g)              # (B, K)
                w = torch.softmax(logits_w, dim=1)    # (B, K)

                mix = (w.unsqueeze(-1) * pb).sum(dim=1)  # (B, C)
                p_true = mix[torch.arange(mix.size(0)), yb]
                loss = -torch.log(p_true.clamp_min(1e-12)).mean()

                opt.zero_grad()
                loss.backward()
                opt.step()

        return self

    def _current_base_prob_list(self, X):
        """
        Текущие вероятности экспертов от фулл-фитов, выровненные и подрезанные по длине.
        """
        aligned_list, min_len = self._collect_and_align(X, self.fitted_, self.classes_)
        return aligned_list, min_len

    def predict_proba(self, X):
        check_is_fitted(self, ["fitted_", "classes_", "gate_", "n_experts_"])
        # X = np.asarray(X, dtype=np.float32)

        # 1) proba экспертов на инференсе
        prob_list, n = self._current_base_prob_list(X)  # список длины K, каждый [n, C]
        K = len(prob_list)
        C = prob_list[0].shape[1]

        # 2) фичи гейтинга на инференсе (можно заменить OOF на текущие proba)
        if self.use_X:
            if hasattr(X, "iloc"):   # DataFrame
                X_slice_num = self._numerify_X(X.iloc[-n:])
            else:                    # ndarray
                X_slice_num = self._numerify_X(X[-n:])
        else:
            X_slice_num = None

        X_slice = (X.iloc[-n:] if hasattr(X, "iloc") else X[-n:])
        G = self._build_gate_features(X_slice_num if X_slice_num is not None else X_slice,
                                    prob_list).astype(np.float32)

        # 3) веса w(x)
        self.gate_.eval()
        with torch.no_grad():
            G_t = torch.tensor(G, dtype=torch.float32, device=self.device)
            logits_w = self.gate_(G_t)              # (n, K)
            w = torch.softmax(logits_w, dim=1).cpu().numpy()  # (n, K)

        # 4) смесь
        mix = np.zeros((n, C), dtype=float)
        for k in range(K):
            mix += w[:, [k]] * prob_list[k]
        return mix

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]

    def predict_weights(self, X):
        """
        Вернёт матрицу весов w(x): shape = (N_common, K).
        Полезно для анализа вклада экспертов.
        """
        check_is_fitted(self, ["fitted_", "classes_", "gate_", "n_experts_"])
        # X = np.asarray(X, dtype=np.float32)
        prob_list, n = self._current_base_prob_list(X)
        
        if self.use_X:
            if hasattr(X, "iloc"):      # DataFrame
                X_slice_num = self._numerify_X(X.iloc[-n:])
                X_slice = X.iloc[-n:]
            else:                       # ndarray
                X_slice_num = self._numerify_X(X[-n:])
                X_slice = X[-n:]
        else:
            X_slice_num = None
            X_slice = X.iloc[-n:] if hasattr(X, "iloc") else X[-n:]

        G = self._build_gate_features(
            X_slice_num if X_slice_num is not None else X_slice,
            prob_list
        ).astype(np.float32)

        self.gate_.eval()
        with torch.no_grad():
            G_t = torch.tensor(G, dtype=torch.float32, device=self.device)
            logits_w = self.gate_(G_t)
            w = torch.softmax(logits_w, dim=1).cpu().numpy()
        return w  # (n, K)

In [69]:
ensemble = AdaptiveMixtureEnsemble(
    classifiers=[pipe_cat, pipe_rule, clf_lstm],
    classes=[-1, 0, 1],          # или None -> объединение
    deepcopy_estimators=True,    # чтобы не перезаписывать исходные инстансы

    use_X=True,                  # добавляем исходные признаки в гейтинг
    use_base_proba=True,         # и OOF / proba базовых
    extra_features="all",        # энтропия и максимум как доп. фичи
    cv=5,

    gate_hidden=64,
    gate_depth=2,
    gate_dropout=0.1,

    epochs=60,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=4096,
    device="cpu",                # или "cuda" если доступна
    random_state=42
)

In [70]:
ensemble.fit(df, df['target'])

AdaptiveMixtureEnsemble(batch_size=4096, classes=[-1, 0, 1],
                        classifiers=[Pipeline(steps=[('tech',
                                                      TechIndicatorsTransformer()),
                                                     ('scaler',
                                                      DataFrameMinMaxScaler()),
                                                     ('cat',
                                                      CatBoostSignalClassifier(cat_params={'depth': 6,
                                                                                           'iterations': 500,
                                                                                           'learning_rate': 0.05,
                                                                                           'loss_function': 'MultiClass',
                                                                                           'random_seed': 42,
                                                                                           'verbose': False},
                                                                               const_margin=0.001))]),
                                     Pipeline(steps...
                                                      RuleSignalClassifier())]),
                                     TimeSeriesLSTMClassifier(device='cpu',
                                                              feature_cols=Index(['close', 'open', 'high', 'low', 'volume', 'USD_RUB', 'close_RUR',
       'RSI_14', 'RSI_14_MA14', 'MACD', 'MACD_Signal', 'MACD_Hist', 'tema',
       'macd', 'macd_signal', 'macd_hist', 'SMA_20', 'SMA_50'],
      dtype='object'),
                                                              max_epochs=25,
                                                              verbose=0)],
                        deepcopy_estimators=True, epochs=60,
                        extra_features='all', gate_dropout=0.1)

In [71]:
ensemble.predict(df)

array([1, 1, 1, ..., 1, 1, 0])

<div class="alert alert-info">

<b>Внимание</b>

В одной из базовых моделей — `TimeSeriesLSTMClassifier` `sklearn.clone` может ругаеться, когда параметры после `__init__` отличаются от переданных. 

У нас меняется/устанавливается feature_cols.
    
</div>

Сделаем класс `TimeSeriesLSTMClassifier` sklearn-совместимым: в `__init__` только присваивания параметров без логики и без мутаций. Любые вычисления и «нормализации» — в `fit`.

In [72]:
class TimeSeriesLSTMClassifier(BaseEstimator, ClassifierMixin):
    """
    sklearn-совместимый LSTM классификатор для временных рядов.
    Совместим с VotingClassifier (поддерживает predict и predict_proba).

    Важно: возвращает предсказания для каждой строки X.
    Для первых (lookback+horizon-1) строк применяется заполнение (pad_strategy):
      - 'edge': повторяется первое доступное предсказание;
      - 'constant': константа pad_value (класс) для первых шагов.
    """
    def __init__(
        self,
        feature_cols: Optional[List[str]] = None,
        target_col: str = "target",
        lookback: int = 64,
        horizon: int = 1,
        hidden_size: int = 64,
        num_layers: int = 2,
        dropout: float = 0.2,
        bidirectional: bool = False,
        batch_size: int = 256,
        max_epochs: int = 30,
        lr: float = 1e-3,
        weight_decay: float = 1e-4,
        patience: int = 7,
        device: Optional[str] = None,
        num_workers: int = 0,
        pad_strategy: str = "edge",   # 'edge' | 'constant'
        pad_value: int = 0,
        verbose: int = 1
    ):
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.lookback = lookback
        self.horizon = horizon
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout
        self.bidirectional = bidirectional
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.lr = lr
        self.weight_decay = weight_decay
        self.patience = patience
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.num_workers = num_workers
        self.pad_strategy = pad_strategy
        self.pad_value = pad_value
        self.verbose = verbose

    def fit(self, X: pd.DataFrame, y=None):
        self.feature_cols = (self.feature_cols
                               if self.feature_cols is not None
                               else list(getattr(X, "columns", range(X.shape[1]))))
        
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame с колонкой 'date' и числовыми фичами.")

        # Сортировка и единый порядок для X и y
        if "date" in X.columns:
            order = np.argsort(pd.to_datetime(X["date"]).values)
            X = X.iloc[order].reset_index(drop=True)
            if y is not None:
                if isinstance(y, (pd.Series, pd.DataFrame, pd.Index)):
                    y = np.asarray(y.iloc[order]).ravel()
                else:
                    y = np.asarray(y).ravel()
        else:
            # если нет date — просто выровняем индексы и приведём y к массиву
            if y is not None and isinstance(y, (pd.Series, pd.DataFrame, pd.Index)):
                y = np.asarray(y).ravel()

        # если y не передан — берём из X[target_col]
        if y is None:
            if self.target_col not in X.columns:
                raise ValueError("y не передан и столбец target_col отсутствует в X.")
            y = X[self.target_col].to_numpy().ravel()

        # фичи
        if self.feature_cols is None:
            feat_cols = _infer_feature_cols(X, self.target_col)
        else:
            feat_cols = list(self.feature_cols)  # важное приведение из pd.Index к list
        
        Xf = X[feat_cols].to_numpy(dtype=np.float32)

        # масштабирование
        self.scaler_ = StandardScaler()
        Xs = self.scaler_.fit_transform(Xf)

        # кодирование меток
        self.le_ = LabelEncoder()
        y_enc = self.le_.fit_transform(np.asarray(y).ravel())
        self.classes_ = self.le_.classes_
        n_classes = len(self.classes_)

        # последовательности
        X_seq, y_seq = _build_sequences(Xs, y_enc, self.lookback, self.horizon)
        ds = _SeqDS(X_seq, y_seq)
        dl = DataLoader(ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

        # модель
        n_features = X_seq.shape[-1]
        self.model_ = _LSTM(
            n_features=n_features, hidden_size=self.hidden_size, num_layers=self.num_layers,
            dropout=self.dropout, bidirectional=self.bidirectional, num_classes=n_classes
        ).to(self.device)
        weights = _class_weights(y_seq, n_classes).to(self.device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        optim = torch.optim.AdamW(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, mode="min", factor=0.5, patience=3, verbose=False)

        best_loss = float("inf")
        best_state = None
        patience_left = self.patience

        for epoch in range(1, self.max_epochs+1):
            self.model_.train()
            losses = []
            for xb, yb in dl:
                xb = xb.to(self.device); yb = yb.to(self.device)
                optim.zero_grad()
                logits = self.model_(xb)
                loss = criterion(logits, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model_.parameters(), 1.0)
                optim.step()
                losses.append(loss.item())
            val_loss = float(np.mean(losses)) if losses else np.nan
            scheduler.step(val_loss)
            if self.verbose:
                print(f"Epoch {epoch:03d} | loss {val_loss:.4f}")
            if val_loss < best_loss - 1e-4:
                best_loss = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.model_.state_dict().items()}
                patience_left = self.patience
            else:
                patience_left -= 1
                if patience_left <= 0:
                    if self.verbose: print("Early stopping")
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state)
        self.feature_cols_ = feat_cols
        self.classes_ = self.le_.classes_
        return self

    def predict(self, X: pd.DataFrame):
        proba = self.predict_proba(X)
        preds_enc = np.argmax(proba, axis=1)
        preds = self.le_.inverse_transform(preds_enc)
        return preds

    def predict_proba(self, X: pd.DataFrame):
        check_is_fitted(self, ["model_", "scaler_", "le_", "feature_cols_"])
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame с колонкой 'date' и числовыми фичами.")
        if "date" in X.columns:
            X = X.sort_values("date").reset_index(drop=True)

        Xf = X[self.feature_cols_].to_numpy(dtype=np.float32)
        Xs = self.scaler_.transform(Xf)

        need = self.lookback + self.horizon - 1
        C = len(self.classes_)

        if len(Xs) < need:
            return np.full((len(Xs), C), 1.0 / C, dtype=np.float32)

        X_seq, _ = _build_sequences(Xs, None, self.lookback, self.horizon)
        ds = _SeqDS(X_seq)
        dl = DataLoader(ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

        self.model_.eval()
        probs_seq = []
        with torch.no_grad():
            for xb in dl:
                xb = xb.to(self.device)
                logits = self.model_(xb)
                p = torch.softmax(logits, dim=1).cpu().numpy()
                probs_seq.append(p)
        probs_seq = np.concatenate(probs_seq, axis=0)  # (len(X)-need+1, C)

        pad_len = need - 1
        if self.pad_strategy == "edge":
            pad_block = np.repeat(probs_seq[[0]], pad_len, axis=0)
        elif self.pad_strategy == "constant":
            pad_block = np.zeros((pad_len, C), dtype=probs_seq.dtype)
            where = np.where(self.classes_ == self.pad_value)[0]
            if len(where) == 0:
                pad_block[:] = 1.0 / C
            else:
                pad_block[:, where[0]] = 1.0
        else:
            raise ValueError("pad_strategy должен быть 'edge' или 'constant'.")

        probs = np.vstack([pad_block, probs_seq])  # (len(X), C)
        return probs

Тестируем класс

In [73]:
clf_lstm = TimeSeriesLSTMClassifier(
    feature_cols=df.columns[1:-1],
    target_col='target',
    lookback=64, horizon=1, max_epochs=25, verbose=0
)

In [74]:
clf_lstm.fit(df, df['target'])  

TimeSeriesLSTMClassifier(device='cpu',
                         feature_cols=Index(['close', 'open', 'high', 'low', 'volume', 'USD_RUB', 'close_RUR',
       'RSI_14', 'RSI_14_MA14', 'MACD', 'MACD_Signal', 'MACD_Hist', 'tema',
       'macd', 'macd_signal', 'macd_hist', 'SMA_20', 'SMA_50'],
      dtype='object'),
                         max_epochs=25, verbose=0)

Собираем ансамбль

In [75]:
ensemble = AdaptiveMixtureEnsemble(
    classifiers=[pipe_cat, pipe_rule, clf_lstm],
    classes=[-1, 0, 1],          # или None -> объединение
    deepcopy_estimators=True,    # чтобы не перезаписывать исходные инстансы

    use_X=True,                  # добавляем исходные признаки в гейтинг
    use_base_proba=True,         # и OOF / proba базовых
    extra_features="all",        # энтропия и максимум как доп. фичи
    cv=5,

    gate_hidden=64,
    gate_depth=2,
    gate_dropout=0.1,

    epochs=60,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=4096,
    device="cpu",                # или "cuda" если доступна
    random_state=42
)

In [76]:
ensemble.fit(df, df['target'])

AdaptiveMixtureEnsemble(batch_size=4096, classes=[-1, 0, 1],
                        classifiers=[Pipeline(steps=[('tech',
                                                      TechIndicatorsTransformer()),
                                                     ('scaler',
                                                      DataFrameMinMaxScaler()),
                                                     ('cat',
                                                      CatBoostSignalClassifier(cat_params={'depth': 6,
                                                                                           'iterations': 500,
                                                                                           'learning_rate': 0.05,
                                                                                           'loss_function': 'MultiClass',
                                                                                           'random_seed': 42,
                                                                                           'verbose': False},
                                                                               const_margin=0.001))]),
                                     Pipeline(steps...
                                                      RuleSignalClassifier())]),
                                     TimeSeriesLSTMClassifier(device='cpu',
                                                              feature_cols=Index(['close', 'open', 'high', 'low', 'volume', 'USD_RUB', 'close_RUR',
       'RSI_14', 'RSI_14_MA14', 'MACD', 'MACD_Signal', 'MACD_Hist', 'tema',
       'macd', 'macd_signal', 'macd_hist', 'SMA_20', 'SMA_50'],
      dtype='object'),
                                                              max_epochs=25,
                                                              verbose=0)],
                        deepcopy_estimators=True, epochs=60,
                        extra_features='all', gate_dropout=0.1)

In [77]:
ensemble.predict(df)

array([0, 0, 0, ..., 1, 1, 1])

In [78]:
ensemble.predict_proba(df)

array([[0., 1., 0.],
       [0., 1., 0.],
       [0., 1., 0.],
       ...,
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.]])

In [79]:
ensemble.predict_weights(df)

array([[0., 1., 0.],
       [0., 1., 0.],
       [0., 1., 0.],
       ...,
       [0., 1., 0.],
       [0., 1., 0.],
       [0., 1., 0.]], dtype=float32)